# ĐỒ ÁN TỐT NGHIỆP: YoFaSoShop_VTON (Ứng dụng TMĐT Thời trang tích hợp Thử đồ ảo)
## NOTEBOOK THỬ NGHIỆM TUẦN 4 - VER2 OOTDIFFUSION-ONLY (21/09 - 27/09/2026)
**Sinh viên thực hiện:** Nguyễn Vũ Khanh - MSSV: 22115053122118 - Lớp: 22T1  
**Giảng viên hướng dẫn:** ThS. Phạm Thị Trà My - Khoa Công nghệ số, ĐH Sư phạm Kỹ thuật - ĐH Đà Nẵng  

---
### Mục tiêu công việc Tuần 4:
1. Chạy thử nghiệm bóc tách phông nền sản phẩm shop bằng **DeepFashion2 (YOLOv8-seg)**, lưu ảnh tách nền vào `results/segmented/` và trích xuất bảng màu chủ đạo (Color Palette).
2. Chạy chuỗi thử đồ ảo **OOTDiffusion VITON-HD** cho `upper_body` và **OOTDiffusion DressCode** cho `lower_body`/`dress` qua GPU đám mây.
3. Xuất và lưu trữ **2 định dạng kết quả** vào `results/tryon/`:
   - **Ảnh thử đồ đơn (Single Try-on):** Chỉ chứa người mẫu sau khi đã ướm trang phục mới (`tryon_result.png`).
   - **Bảng đối sánh 3 cột:** `[Người mẫu gốc | Sản phẩm bóc nền | Kết quả thử đồ AI]` (`week4_report_comparison.jpg`).
4. Tự động đóng gói và tải kết quả về máy tính cục bộ khi chạy trên Google Colab.


### Bước 1: Kiểm tra GPU Google Colab

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q -U ultralytics diffusers transformers accelerate onnxruntime opencv-python scikit-learn pillow matplotlib huggingface_hub gradio_client segmentation-models-pytorch safetensors


### Bước 3: Tải trọng số DeepFashion2 YOLOv8-seg và khởi tạo thư mục dữ liệu


In [ ]:
import os, sys
from pathlib import Path

# 1. Tự động định vị Root directory chuẩn của Repo (Hỗ trợ cả Google Colab và Local VS Code)
curr_dir = os.getcwd()
if curr_dir.endswith(os.path.join("ml-training", "notebooks")) or curr_dir.endswith("notebooks"):
    # Đang mở từ thư mục notebooks cục bộ trên Windows/Linux -> Chuyển về gốc Repo
    project_root = Path(curr_dir).resolve().parents[1]
    os.chdir(project_root)
    print(f"[*] Đã chuyển thư mục làm việc về gốc Repo: {os.getcwd()}")
else:
    print(f"[*] Thư mục làm việc hiện tại: {os.getcwd()}")

# 2. Khởi tạo cấu trúc cây thư mục chuẩn cho cả Colab và Local
dirs_to_create = [
    "data/test_samples/garments/tops",
    "data/test_samples/garments/bottoms",
    "data/test_samples/garments/dresses",
    "data/test_samples/persons",
    "data/test_samples/results/segmented",
    "data/test_samples/results/tryon",
    "weights"
]
for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)
print("✅ Các thư mục lưu trữ dữ liệu và kết quả đã sẵn sàng!")

# 3. Tải trọng số YOLOv8s-seg DeepFashion2 nếu chưa có
weight_path = "weights/deepfashion2_yolov8s-seg.pt"
if not os.path.exists(weight_path) or os.path.getsize(weight_path) < 100000:
    print("Đang tải trọng số YOLOv8s-seg DeepFashion2 (~24MB)...")
    !wget -q -O weights/deepfashion2_yolov8s-seg.pt https://huggingface.co/Bingsu/adetailer/resolve/main/deepfashion2_yolov8s-seg.pt
    print("Tải hoàn tất trọng số!")
else:
    print("Trọng số YOLOv8s-seg đã sẵn sàng!")


### Bước 4: Upload ảnh trang phục, người mẫu & File/Ảnh tham chiếu form dáng

Quy trình upload thông minh gồm 4 thành phần:
1. **File tham chiếu `product_analysis.md` (Tùy chọn - Khuyến nghị):** Khóa chính xác category, chống nhầm áo thành dress.
2. **Ảnh trang phục shop:** Ảnh sản phẩm cần thử (packshot hoặc trải phẳng).
3. **Ảnh người mẫu shop mặc sản phẩm (Tùy chọn - Rất khuyến nghị):** Dùng làm mốc tham chiếu đo đạc độ dài tay áo, thân áo/váy/quần chuẩn thực tế khi mặc trên người mẫu shop, giúp AI ưu tiên chọn candidate có độ dài tay áo chuẩn xác nhất (tránh bị co cụt tay áo khi khách hàng mặc áo cộc tay).
4. **Ảnh khách hàng / người mẫu cần thử đồ:** Người mặc trang phục ảo.


In [ ]:
import io
import os
import re
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

# Bước 4 tự tạo thư mục để vẫn chạy được nếu runtime Colab vừa khởi động lại.
garment_dir = Path("data/test_samples/garments/uploaded")
person_dir = Path("data/test_samples/persons/uploaded")
ref_dir = Path("data/test_samples/reference_analysis")
shop_model_dir = Path("data/test_samples/shop_models/uploaded")

for d in (garment_dir, person_dir, ref_dir, shop_model_dir):
    d.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Cell upload này cần chạy trên Google Colab.") from exc

print("[*] Đang nạp mô hình kiểm tra ảnh và tư thế...")
fashion_detector = YOLO("weights/deepfashion2_yolov8s-seg.pt")
try:
    person_detector = YOLO("yolo26s-pose.pt")
    print("✅ Đã nạp thành công YOLO26s-Pose thế hệ mới!")
except Exception as e:
    print(f"⚠️ Không tải được yolo26s-pose.pt ({e}), tự động fallback về yolov8n-pose.pt")
    person_detector = YOLO("yolov8n-pose.pt")


def read_uploaded_image(filename, file_bytes):
    """Đọc và chuẩn hóa mọi định dạng ảnh Pillow hỗ trợ sang RGB."""
    try:
        with Image.open(io.BytesIO(file_bytes)) as probe:
            probe.verify()
        image = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return None, f"'{filename}' không phải file ảnh hợp lệ ({exc})"

    if image.width < 128 or image.height < 128:
        return None, f"'{filename}' quá nhỏ ({image.width}x{image.height}); cần ít nhất 128x128 px"
    return image, None


POSE_KEYPOINT_THRESHOLD = 0.35


def classify_person_frame(keypoint_scores):
    """Phân loại khung người từ 17 keypoint COCO của YOLO Pose."""
    if keypoint_scores is None or len(keypoint_scores) < 17:
        return "invalid", {"shoulders": 0, "hips": 0, "knees": 0, "ankles": 0}

    visible = keypoint_scores >= POSE_KEYPOINT_THRESHOLD
    counts = {
        "shoulders": int(visible[[5, 6]].sum()),
        "hips": int(visible[[11, 12]].sum()),
        "knees": int(visible[[13, 14]].sum()),
        "ankles": int(visible[[15, 16]].sum()),
    }

    upper_core_visible = counts["shoulders"] >= 1 and counts["hips"] >= 1
    knees_visible = counts["knees"] >= 1
    ankles_visible = counts["ankles"] >= 1
    if upper_core_visible and knees_visible and ankles_visible:
        return "full_body", counts
    if upper_core_visible and knees_visible:
        return "three_quarter", counts
    if upper_core_visible:
        return "half_body", counts
    return "invalid", counts


def best_person_detection(image):
    result = person_detector.predict(
        source=np.asarray(image), classes=[0], conf=0.25, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
        return 0.0, 0.0, 0, "invalid", {}

    confidences = result.boxes.conf.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()
    image_area = image.width * image.height
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    best_index = int(np.argmax(confidences))
    keypoint_confidences = result.keypoints.conf
    visible_keypoints = 0
    person_frame = "invalid"
    frame_keypoints = {}
    if keypoint_confidences is not None:
        keypoint_scores = keypoint_confidences[best_index].cpu().numpy()
        visible_keypoints = int((keypoint_scores >= POSE_KEYPOINT_THRESHOLD).sum())
        person_frame, frame_keypoints = classify_person_frame(keypoint_scores)
    return (
        float(confidences[best_index]),
        float(areas[best_index] / image_area),
        visible_keypoints,
        person_frame,
        frame_keypoints,
    )


def best_garment_detection(image):
    result = fashion_detector.predict(
        source=np.asarray(image), conf=0.25, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0

    confidences = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    best_index = int(np.argmax(confidences))
    class_id = int(classes[best_index])
    label = result.names[class_id] if isinstance(result.names, dict) else result.names[class_id]
    return str(label), float(confidences[best_index])


def validate_image_role(image, expected_role):
    if expected_role == "person":
        (
            person_confidence,
            person_area_ratio,
            visible_keypoints,
            person_frame,
            frame_keypoints,
        ) = best_person_detection(image)
        frame_summary = ", ".join(
            f"{name}={count}/2" for name, count in frame_keypoints.items()
        ) or "không có keypoint"
        if (
            person_confidence < 0.35
            or person_area_ratio < 0.08
            or visible_keypoints < 5
            or person_frame == "invalid"
        ):
            return False, (
                "không nhận diện được người đủ rõ "
                f"(confidence={person_confidence:.2f}, vùng người={person_area_ratio:.0%}, "
                f"keypoints={visible_keypoints}/17, {frame_summary}). "
                "Hãy dùng ảnh chính diện và thấy rõ ít nhất từ vai đến hông."
            ), None
        return True, (
            f"đã nhận diện person (confidence={person_confidence:.2f}, "
            f"keypoints={visible_keypoints}/17, frame={person_frame}, {frame_summary})"
        ), person_frame

    garment_label, garment_confidence = best_garment_detection(image)
    if garment_label is None or garment_confidence < 0.25:
        return False, "không nhận diện được quần áo bằng DeepFashion2", None
    return True, (
        f"đã nhận diện garment '{garment_label}' "
        f"(confidence={garment_confidence:.2f}); Bước 5 sẽ tách riêng trang phục bằng mask"
    ), garment_label


def safe_stem(filename):
    stem = re.sub(r"[^0-9A-Za-z_-]+", "_", Path(filename).stem).strip("_")
    return stem[:80] or "uploaded_image"


def parse_product_analysis(text):
    """Trích xuất metadata đã giải từ file product_analysis.md hoặc .json."""
    metadata = {}
    if not text:
        return None

    json_match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if json_match:
        try:
            metadata.update(json.loads(json_match.group(1)))
        except Exception:
            pass
    elif text.strip().startswith("{") and text.strip().endswith("}"):
        try:
            metadata.update(json.loads(text))
        except Exception:
            pass

    if not metadata.get("resolved_label"):
        m = re.search(r"TieuLoaiTrangPhuc[`\s:]+([a-z0-9_]+)", text, re.IGNORECASE)
        if m:
            metadata["resolved_label"] = m.group(1).strip()

    if not metadata.get("vton_category"):
        m = re.search(r"PhanLoaiVTON[`\s:]+([a-z0-9_]+)", text, re.IGNORECASE)
        if m:
            metadata["vton_category"] = m.group(1).strip()

    if not metadata.get("resolved_label"):
        m = re.search(r"Category:\s*\*\*(.*?)\*\*.*?`([a-z0-9_]+)`", text, re.IGNORECASE)
        if m:
            metadata["category"] = m.group(1).strip()
            metadata["resolved_label"] = m.group(2).strip()

    if not metadata.get("vton_category"):
        m = re.search(r"VTON\s*category:\s*\*\*([a-z0-9_]+)\*\*", text, re.IGNORECASE)
        if m:
            metadata["vton_category"] = m.group(1).strip()

    return metadata if (metadata.get("resolved_label") or metadata.get("vton_category")) else None


def upload_product_analysis_reference(destination_dir):
    """Cho phép tải file product_analysis.md để khóa category, chống phân loại sai."""
    print("\n👉 [BƯỚC 4.1 - KHUYẾN NGHỊ] Tải file 'product_analysis.md' (hoặc .json) từ notebook phân tích sản phẩm:")
    print("   💡 File này chứa category đã được kiểm định chính xác, giúp khóa đúng loại áo/quần/váy, tránh lỗi DeepFashion2 nhận diện nhầm áo thành dress.")
    print("   💡 Nếu chưa có file tham chiếu này, bạn chỉ cần bấm 'Cancel' (Hủy) để tiếp tục quy trình tự động.")
    try:
        uploaded = files.upload()
    except Exception as exc:
        print(f"ℹ️ Bỏ qua bước tải tham chiếu: {exc}")
        return None

    if not uploaded:
        print("ℹ️ Bạn đã bỏ qua tải file tham chiếu. Hệ thống sẽ nhận diện tự động bằng DeepFashion2.")
        return None

    for filename, file_bytes in uploaded.items():
        try:
            content = file_bytes.decode("utf-8", errors="replace")
        except Exception:
            continue
        meta = parse_product_analysis(content)
        if meta and (meta.get("resolved_label") or meta.get("vton_category")):
            save_path = destination_dir / f"ref_{safe_stem(filename)}.md"
            save_path.write_text(content, encoding="utf-8")
            label_disp = meta.get("category") or meta.get("resolved_label", "n/a")
            print(f"✅ Đã nạp thành công tham chiếu từ file '{filename}':")
            print(f"   🎯 Loại trang phục chuẩn : {meta.get('resolved_label')} ({label_disp})")
            print(f"   🎯 Phân loại VTON chuẩn : {meta.get('vton_category')}")
            return meta
        else:
            print(f"⚠️ File '{filename}' không chứa cấu trúc metadata hợp lệ từ product_reference_analysis.")

    print("ℹ️ Tiếp tục quy trình với nhận diện tự động từ DeepFashion2.")
    return None


def upload_shop_model_reference(destination_dir):
    """Cho phép tải ảnh người mẫu của shop mặc sản phẩm để đo form/độ dài tay áo, vạt áo."""
    print("\n👉 [BƯỚC 4.3 - KHUYẾN NGHỊ] Tải ảnh NGƯỜI MẪU SHOP mặc sản phẩm (Lookbook Model Reference):")
    print("   💡 Dùng làm chuẩn tham chiếu giải phẫu học: đo tỷ lệ chiều dài tay áo, gấu áo/váy/quần thực tế khi mặc lên người mẫu shop.")
    print("   💡 Giúp AI giải quyết triệt để lỗi tay áo bị co cụt khi khách hàng đang mặc áo cộc nách/sát nách.")
    print("   💡 Nếu chưa có ảnh này, bạn chỉ cần bấm 'Cancel' (Hủy) để tiếp tục quy trình tự động.")
    try:
        uploaded = files.upload()
    except Exception as exc:
        print(f"ℹ️ Bỏ qua bước tải ảnh người mẫu shop: {exc}")
        return None

    if not uploaded:
        print("ℹ️ Bạn đã bỏ qua tải ảnh người mẫu shop. Tiếp tục phân tích theo ảnh sản phẩm.")
        return None

    for filename, file_bytes in uploaded.items():
        image, image_error = read_uploaded_image(filename, file_bytes)
        if image_error:
            print(f"❌ {image_error}")
            continue
        save_path = destination_dir / f"shop_model_{safe_stem(filename)}.png"
        image.save(save_path, format="PNG")
        print(f"✅ Đã nạp ảnh người mẫu shop tham chiếu: {save_path.name}")
        return str(save_path)

    return None


def analyze_model_garment_fit(image_path_or_img, garment_cat="upper_body"):
    """Đo tỷ lệ chiều dài tay áo và gấu trang phục trên người mẫu bằng Pose + DeepFashion2."""
    if image_path_or_img is None:
        return None
    try:
        if isinstance(image_path_or_img, (str, Path)):
            img = Image.open(image_path_or_img).convert("RGB")
        else:
            img = image_path_or_img.convert("RGB")
        img_np = np.asarray(img)
        h, w = img_np.shape[:2]

        p_res = person_detector.predict(img_np, verbose=False)[0]
        if p_res.keypoints is None or len(p_res.keypoints.xy) == 0:
            return None
        kpts = p_res.keypoints.xy[0].cpu().numpy()
        confs = p_res.keypoints.conf[0].cpu().numpy() if p_res.keypoints.conf is not None else np.ones(len(kpts))

        f_res = fashion_detector.predict(img_np, conf=0.20, verbose=False)[0]
        if f_res.boxes is None or len(f_res.boxes) == 0 or f_res.masks is None:
            return None

        masks = f_res.masks.data.cpu().numpy()
        mask = cv2.resize(masks[0], (w, h), interpolation=cv2.INTER_NEAREST)
        ys, xs = np.where(mask > 0.5)
        if len(xs) < 50:
            return None

        sleeve_ratios = []
        for s_idx, e_idx in ((5, 7), (6, 8)):
            if confs[s_idx] >= 0.25 and confs[e_idx] >= 0.25:
                sy, sx = kpts[s_idx][1], kpts[s_idx][0]
                ey, ex = kpts[e_idx][1], kpts[e_idx][0]
                arm_span = max(10.0, ey - sy)
                arm_side_ys = ys[xs < sx + 15] if s_idx == 5 else ys[xs > sx - 15]
                if len(arm_side_ys):
                    sleeve_hem_y = float(arm_side_ys.max())
                    sleeve_length = max(0.0, sleeve_hem_y - sy)
                    sleeve_ratios.append(min(1.5, sleeve_length / arm_span))

        avg_sleeve_ratio = float(np.mean(sleeve_ratios)) if sleeve_ratios else 0.5
        hem_y = float(ys.max())
        shoulder_y = float(np.mean(kpts[[5, 6], 1])) if min(confs[[5, 6]]) >= 0.25 else float(ys.min())
        hip_y = float(np.mean(kpts[[11, 12], 1])) if min(confs[[11, 12]]) >= 0.25 else shoulder_y + 100.0
        torso_span = max(10.0, hip_y - shoulder_y)
        hem_ratio = (hem_y - shoulder_y) / torso_span

        return {
            "sleeve_ratio": round(avg_sleeve_ratio, 3),
            "hem_ratio": round(hem_ratio, 3),
            "garment_bbox": [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
        }
    except Exception as exc:
        print(f"⚠️ Đo form người mẫu không hoàn tất: {exc}")
        return None


def upload_validated_image(expected_role, destination_dir):
    role_name = "TRANG PHỤC" if expected_role == "garment" else "NGƯỜI MẪU"
    print(f"\n👉 Chọn ảnh {role_name}; tên file có thể tùy ý (PNG/JPG/JPEG/WEBP...):")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError(f"Chưa có file nào được chọn cho {role_name}.")

    rejected = []
    for filename, file_bytes in uploaded.items():
        image, image_error = read_uploaded_image(filename, file_bytes)
        if image_error:
            print(f"❌ {image_error}")
            rejected.append(filename)
            continue

        accepted, message, detected_label = validate_image_role(image, expected_role)
        if not accepted:
            print(f"❌ Từ chối '{filename}': {message}.")
            rejected.append(filename)
            continue

        destination = destination_dir / f"{expected_role}_{safe_stem(filename)}.png"
        image.save(destination, format="PNG")
        print(f"✅ Chấp nhận '{filename}': {message}.")
        print(f"   Đã chuẩn hóa và lưu tại: {destination}")
        return str(destination), detected_label

    raise ValueError(
        f"Không có ảnh {role_name} hợp lệ. Hãy chọn đúng loại ảnh rồi chạy lại Bước 4. "
        f"Các file đã từ chối: {', '.join(rejected)}"
    )


# 1. TẢI FILE THAM CHIẾU (NẾU CÓ - KHUYẾN NGHỊ ĐỂ TRÁNH NHẦM ÁO THÀNH DRESS)
reference_metadata = upload_product_analysis_reference(ref_dir)

# 2. TẢI ẢNH TRANG PHỤC SHOP
garment_path, detected_garment_class = upload_validated_image("garment", garment_dir)

# 3. [MỚI] TẢI ẢNH NGƯỜI MẪU SHOP MẶC SẢN PHẨM (NẾU CÓ - THAM CHIẾU FORM & ĐỘ DÀI TAY ÁO/GẤU VÁY)
shop_model_path = upload_shop_model_reference(shop_model_dir)
shop_model_fit = analyze_model_garment_fit(shop_model_path) if shop_model_path else None

# 4. TẢI ẢNH KHÁCH HÀNG / NGƯỜI CẦN THỬ ĐỒ
person_path, person_frame = upload_validated_image("person", person_dir)
user_garment_fit = analyze_model_garment_fit(person_path)

# Phân tích sơ bộ tư thế tay người mẫu
try:
    _person_img = Image.open(person_path).convert("RGB")
    _pose_res = person_detector.predict(np.asarray(_person_img), conf=0.20, verbose=False)[0]
    if _pose_res.keypoints is not None and len(_pose_res.keypoints.xy):
        _kpts = _pose_res.keypoints.xy[0].cpu().numpy()
        _confs = _pose_res.keypoints.conf[0].cpu().numpy() if _pose_res.keypoints.conf is not None else np.ones(len(_kpts))
        if len(_kpts) >= 11:
            _wrist_confs = _confs[[9, 10]]
            _is_behind = False
            if np.mean(_wrist_confs) < 0.35 or (_confs[9] < 0.35 and _confs[10] < 0.35):
                _is_behind = True
            elif len(_kpts) >= 13 and min(_confs[[11, 12]]) >= 0.25:
                _hx1, _hx2 = min(_kpts[11][0], _kpts[12][0]), max(_kpts[11][0], _kpts[12][0])
                _hspan = max(10.0, _hx2 - _hx1)
                if (_hx1 - 0.25 * _hspan <= _kpts[9][0] <= _hx2 + 0.25 * _hspan) and                    (_hx1 - 0.25 * _hspan <= _kpts[10][0] <= _hx2 + 0.25 * _hspan) and np.mean(_wrist_confs) < 0.55:
                    _is_behind = True
            if _is_behind:
                print("⚠️ [LƯU Ý TƯ THẾ NGƯỜI MẪU]: Phát hiện hai tay đang để ra sau lưng hoặc bị che khuất.")
                print("   💡 Mẹo Virtual Try-On: Để kết quả thử đồ giữ trọn vẹn 100% dáng tay và form chuẩn,")
                print("   khuyến nghị dùng ảnh người mẫu đứng thẳng, hai tay buông tự nhiên hai bên thân người.")
except Exception:
    pass

UPPER_BODY_LABELS = {
    "short_sleeved_shirt", "long_sleeved_shirt", "short_sleeved_outwear",
    "long_sleeved_outwear", "vest", "sling",
}
LOWER_BODY_LABELS = {"shorts", "trousers", "skirt"}
DRESS_LABELS = {
    "short_sleeved_dress", "long_sleeved_dress", "vest_dress", "sling_dress",
}
SUPPORTED_GARMENT_LABELS = UPPER_BODY_LABELS | LOWER_BODY_LABELS | DRESS_LABELS


def garment_category_from_label(label):
    normalized = (label or "").strip().lower()
    if normalized in UPPER_BODY_LABELS:
        return "upper_body"
    if normalized in LOWER_BODY_LABELS:
        return "lower_body"
    if normalized in DRESS_LABELS:
        return "dress"
    return None


# 5. GIẢI QUYẾT CATEGORY: ƯU TIÊN FILE THAM CHIẾU PRODUCT_ANALYSIS.MD
if reference_metadata and reference_metadata.get("resolved_label"):
    garment_class_name = reference_metadata["resolved_label"]
    garment_category = reference_metadata.get("vton_category") or garment_category_from_label(garment_class_name)
    garment_description = reference_metadata.get("category") or garment_class_name.replace("_", " ")
    category_source = "product_analysis.md (Verified Catalog)"
    print(f"\n🔒 ĐÃ KHÓA CATEGORY THEO FILE THAM CHIẾU: {garment_class_name} ({garment_description}) -> {garment_category}")
    if detected_garment_class != garment_class_name:
        print(f"   ⚠️ Đã vô hiệu hóa nhận diện sơ bộ '{detected_garment_class}' của DeepFashion2 để chống nhầm áo thành dress!")
else:
    garment_class_name = detected_garment_class
    garment_description = (garment_class_name or "clothing").replace("_", " ")
    garment_category = garment_category_from_label(garment_class_name)
    category_source = "DeepFashion2 raw detection"

if garment_category is None:
    raise ValueError(
        f"Nhãn garment '{garment_class_name}' không thuộc 13 nhãn DeepFashion2 được hỗ trợ. "
        "Notebook đã dừng trước khi gọi GPU."
    )

print("\n--- THÔNG TIN DỮ LIỆU ĐÃ KIỂM TRA THÀNH CÔNG ---")
print(f"[+] Trang phục : {garment_path} ({os.path.getsize(garment_path)/1024:.1f} KB)")
print(f"[+] Loại đồ    : {garment_description} (Nguồn: {category_source})")
print(f"[+] Vùng try-on: {garment_category}")
print(f"[+] Người mẫu  : {person_path} ({os.path.getsize(person_path)/1024:.1f} KB)")
print(f"[+] Khung người: {person_frame}")

if shop_model_fit:
    print(f"\n📏 [FORM MẪU SHOP] Tỷ lệ tay áo chuẩn: {shop_model_fit['sleeve_ratio']:.0%} bắp tay | Gấu áo: {shop_model_fit['hem_ratio']:.2f} thân")
    if user_garment_fit:
        print(f"📏 [ÁO USER ĐANG MẶC] Tỷ lệ tay áo user: {user_garment_fit['sleeve_ratio']:.0%} bắp tay")
        if shop_model_fit['sleeve_ratio'] > user_garment_fit['sleeve_ratio'] + 0.15:
            print("   ⚠️ LƯU Ý KỸ THUẬT: Áo shop có tay áo dài hơn áo user đang mặc!")
            print("   -> Hệ thống sẽ kích hoạt bộ lọc độ dài tay áo ở Bước 6 để ưu tiên candidate giữ được tay áo dài của shop.")

# Hiển thị trực quan (3 ảnh nếu có mẫu shop, 2 ảnh nếu không có)
if shop_model_path:
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(Image.open(garment_path))
    ax[0].set_title(f"1. Sản phẩm shop\n{Path(garment_path).name}\n({garment_class_name})", fontsize=11)
    ax[0].axis("off")
    ax[1].imshow(Image.open(shop_model_path))
    ax[1].set_title(f"2. Người mẫu shop (Tham chiếu form)\nTay áo: {shop_model_fit['sleeve_ratio']:.0%} bắp tay" if shop_model_fit else "2. Người mẫu shop", fontsize=11)
    ax[1].axis("off")
    ax[2].imshow(Image.open(person_path))
    ax[2].set_title(f"3. Khách hàng: {Path(person_path).name}\nKhung: {person_frame}", fontsize=11)
    ax[2].axis("off")
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(Image.open(garment_path))
    ax[0].set_title(f"Trang phục: {Path(garment_path).name}\nLoại: {garment_class_name} ({garment_category})", fontsize=12)
    ax[0].axis("off")
    ax[1].imshow(Image.open(person_path))
    ax[1].set_title(f"Người mẫu: {Path(person_path).name}\nKhung: {person_frame}", fontsize=12)
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()


### Bước 5: Chạy thử nghiệm bóc tách nền sản phẩm (DeepFashion2 YOLOv8-seg, Tinh chỉnh Studio-aware & Color Palette)


In [ ]:
from ultralytics import YOLO
import cv2, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

model = YOLO("weights/deepfashion2_yolov8s-seg.pt")
print("Nạp YOLOv8s-seg thành công!")
print(f"Đang phân đoạn và tách nền: {garment_path}")

# Đảm bảo thư mục lưu trữ tồn tại
out_segmented_dir = "data/test_samples/results/segmented"
os.makedirs(out_segmented_dir, exist_ok=True)
out_nobg_path = os.path.join(out_segmented_dir, "garment_nobg.png")
out_mask_path = os.path.join(out_segmented_dir, "garment_mask.png")
out_isolated_path = os.path.join(out_segmented_dir, "garment_isolated_white.png")

# ==============================================================================
# HÀM BỔ TRỢ TINH CHỈNH MASK & BÓC NỀN STUDIO (Kế thừa từ product_reference_analysis)
# ==============================================================================
UPPER_LABELS = {
    "short_sleeved_shirt", "long_sleeved_shirt", "short_sleeved_outwear",
    "long_sleeved_outwear", "vest", "sling",
}
LOWER_LABELS = {"shorts", "trousers", "skirt"}
DRESS_LABELS = {
    "short_sleeved_dress", "long_sleeved_dress", "vest_dress", "sling_dress",
}

def _foreground_runs(row):
    padded = np.pad(row.astype(np.int8), (1, 1))
    changes = np.diff(padded)
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0] - 1
    return list(zip(starts.tolist(), ends.tolist()))

def _dress_lower_branch_cleanup(mask, label):
    """Loại nhánh nền lớn nằm hai bên thân váy nhưng giữ tà trung tâm và váy xòe liền khối."""
    if label not in DRESS_LABELS:
        return mask.copy(), {"cleanup_applied": False}
    ys, xs = np.where(mask)
    if len(xs) < 500:
        return mask.copy(), {"cleanup_applied": False}
    x1, x2, y1, y2 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())
    garment_height = max(1, y2 - y1 + 1)
    garment_width = max(1, x2 - x1 + 1)
    split_ratio = 0.60 if label == "long_sleeved_dress" else 0.38
    split_y = min(y2, int(round(y1 + split_ratio * garment_height)))
    band_start = max(y1, split_y - max(6, int(0.07 * garment_height)))
    bbox_center = (x1 + x2) / 2.0
    anchor_centers, anchor_widths = [], []
    for row_y in range(band_start, split_y):
        runs = _foreground_runs(mask[row_y])
        if not runs:
            continue
        run = min(runs, key=lambda pair: abs(((pair[0] + pair[1]) / 2.0) - bbox_center))
        anchor_centers.append((run[0] + run[1]) / 2.0)
        anchor_widths.append(run[1] - run[0] + 1)
    if not anchor_centers:
        return mask.copy(), {"cleanup_applied": False}
    anchor_center = float(np.median(anchor_centers))
    anchor_width = max(8.0, float(np.percentile(anchor_widths, 40)))
    lower = np.zeros_like(mask, dtype=np.uint8)
    lower[split_y:y2 + 1] = mask[split_y:y2 + 1].astype(np.uint8)
    kernel_size = max(3, min(7, int(round(min(garment_width, garment_height) * 0.004))))
    if kernel_size % 2 == 0:
        kernel_size += 1
    opened = cv2.morphologyEx(
        lower, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    )
    component_total, labels, stats, centroids = cv2.connectedComponentsWithStats(opened, 8)
    if component_total <= 2:
        return mask.copy(), {"cleanup_applied": False}
    central_left = anchor_center - max(anchor_width * 0.90, garment_width * 0.10)
    central_right = anchor_center + max(anchor_width * 0.90, garment_width * 0.10)
    minimum_component_area = max(64, int(mask.sum() * 0.002))
    keep_indices = []
    for component_index in range(1, component_total):
        area = int(stats[component_index, cv2.CC_STAT_AREA])
        if area < minimum_component_area:
            continue
        comp_x = int(stats[component_index, cv2.CC_STAT_LEFT])
        comp_width = int(stats[component_index, cv2.CC_STAT_WIDTH])
        comp_right = comp_x + comp_width - 1
        comp_center = float(centroids[component_index][0])
        overlap = max(0.0, min(comp_right, central_right) - max(comp_x, central_left) + 1)
        overlap_ratio = overlap / max(comp_width, 1)
        center_distance = abs(comp_center - anchor_center)
        if (comp_x <= anchor_center <= comp_right or overlap_ratio >= 0.30 or
            center_distance <= max(anchor_width * 0.80, garment_width * 0.13)):
            keep_indices.append(component_index)
    if not keep_indices:
        return mask.copy(), {"cleanup_applied": False}
    kept_lower = np.isin(labels, keep_indices)
    refined = mask.copy()
    refined[split_y:y2 + 1] = kept_lower[split_y:y2 + 1]
    return refined, {"cleanup_applied": True}

def _remove_distant_dress_components(mask, label):
    """Bỏ component lớn xa thân chính váy."""
    if label not in DRESS_LABELS:
        return mask.copy(), {"component_cleanup_applied": False}
    binary = mask.astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(opened, 8)
    if count <= 2:
        return mask.copy(), {"component_cleanup_applied": False}
    component_indices = list(range(1, count))
    main_index = max(component_indices, key=lambda idx: int(stats[idx, cv2.CC_STAT_AREA]))
    main_x = int(stats[main_index, cv2.CC_STAT_LEFT])
    main_y = int(stats[main_index, cv2.CC_STAT_TOP])
    main_width = int(stats[main_index, cv2.CC_STAT_WIDTH])
    main_height = int(stats[main_index, cv2.CC_STAT_HEIGHT])
    main_right = main_x + main_width - 1
    main_center_x = float(centroids[main_index][0])
    ys, xs = np.where(mask)
    overall_x1, overall_x2 = int(xs.min()), int(xs.max())
    overall_y1, overall_y2 = int(ys.min()), int(ys.max())
    overall_width = max(1, overall_x2 - overall_x1 + 1)
    overall_height = max(1, overall_y2 - overall_y1 + 1)
    minimum_area = max(64, int(mask.sum() * 0.001))
    keep_indices = [main_index]
    for component_index in component_indices:
        if component_index == main_index:
            continue
        area = int(stats[component_index, cv2.CC_STAT_AREA])
        comp_x = int(stats[component_index, cv2.CC_STAT_LEFT])
        comp_y = int(stats[component_index, cv2.CC_STAT_TOP])
        comp_width = int(stats[component_index, cv2.CC_STAT_WIDTH])
        comp_height = int(stats[component_index, cv2.CC_STAT_HEIGHT])
        comp_right = comp_x + comp_width - 1
        comp_bottom = comp_y + comp_height - 1
        comp_center_x = float(centroids[component_index][0])
        horizontal_gap = max(0, main_x - comp_right, comp_x - main_right)
        central_distance_limit = max(overall_width * 0.16, min(main_width * 0.30, overall_width * 0.22))
        near_center = abs(comp_center_x - main_center_x) <= central_distance_limit
        plausible_sleeve = (
            comp_y <= overall_y1 + 0.28 * overall_height
            and comp_bottom <= overall_y1 + 0.65 * overall_height
            and horizontal_gap <= 0.10 * overall_width
        )
        if area >= minimum_area and (near_center or plausible_sleeve):
            keep_indices.append(component_index)
    kept_seed = np.isin(labels, keep_indices).astype(np.uint8)
    restored = (cv2.dilate(kept_seed, kernel, iterations=1) > 0) & mask
    return restored, {"component_cleanup_applied": True}

def _trouser_gap_profile(raw_mask):
    """Đo khoảng trống giữa hai ống quần làm topology prior."""
    ys, xs = np.where(raw_mask)
    if len(xs) < 200:
        return None
    x1, x2, y1, y2 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())
    width = max(1, x2 - x1 + 1)
    height = max(1, y2 - y1 + 1)
    center = (x1 + x2) / 2.0
    minimum_run = max(3, int(width * 0.08))
    rows = []
    for row_y in range(int(y1 + 0.18 * height), y2 + 1):
        runs = [run for run in _foreground_runs(raw_mask[row_y]) if run[1] - run[0] + 1 >= minimum_run]
        left_runs = [run for run in runs if run[1] < center]
        right_runs = [run for run in runs if run[0] > center]
        if not left_runs or not right_runs:
            continue
        left = max(left_runs, key=lambda run: run[1])
        right = min(right_runs, key=lambda run: run[0])
        gap_left, gap_right = left[1] + 1, right[0] - 1
        gap_width = gap_right - gap_left + 1
        if 1 <= gap_width <= 0.38 * width:
            rows.append((row_y, gap_left, gap_right))
    if len(rows) < max(6, int(height * 0.035)):
        return None
    return {"rows": rows, "bbox": [x1, y1, x2, y2], "center": center, "width": width, "height": height}

def _solidify_light_trousers(refined_mask, raw_mask, lab_image, label):
    """Vá phần rỗng bên trong quần sáng màu mà không làm nở biên ngoài."""
    if label not in {"trousers", "shorts"}:
        return refined_mask.copy(), {"trouser_topology_applied": False}
    refined = refined_mask.astype(bool)
    raw = raw_mask.astype(bool)
    profile = _trouser_gap_profile(raw)
    if profile is None:
        return refined_mask.copy(), {"trouser_topology_applied": False}
    candidate = (refined | raw).astype(np.uint8)
    rows = profile["rows"]
    row_y = np.array([item[0] for item in rows], dtype=float)
    gap_left = np.array([item[1] for item in rows], dtype=float)
    gap_right = np.array([item[2] for item in rows], dtype=float)
    x1, y1, x2, raw_bottom = profile["bbox"]
    candidate_bool = candidate.astype(bool)
    row_span_guard = np.zeros_like(candidate_bool)
    for y in range(candidate_bool.shape[0]):
        xs = np.where(candidate_bool[y])[0]
        if len(xs):
            row_span_guard[y, int(xs.min()):int(xs.max()) + 1] = True
    repair_width = max(5, min(51, int(round(profile["width"] * 0.08)) | 1))
    repair_height = max(7, min(41, int(round(profile["height"] * 0.025)) | 1))
    horizontal_close = cv2.morphologyEx(
        candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (repair_width, 3))
    ).astype(bool)
    vertical_close = cv2.morphologyEx(
        candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (3, repair_height))
    ).astype(bool)
    repair_roi = np.zeros_like(candidate_bool)
    repair_roi[max(y1, int(row_y.min()) - repair_height):, x1:x2 + 1] = True
    repair_candidates = (horizontal_close | vertical_close) & ~candidate_bool & row_span_guard & repair_roi
    repair_count, repair_labels, repair_stats, _ = cv2.connectedComponentsWithStats(
        repair_candidates.astype(np.uint8), connectivity=8
    )
    approved_repair = np.zeros_like(candidate_bool)
    candidate_area = max(1, int(candidate_bool.sum()))
    maximum_repair_area = max(48, int(candidate_area * 0.025))
    for component_id in range(1, repair_count):
        area = int(repair_stats[component_id, cv2.CC_STAT_AREA])
        if area <= maximum_repair_area:
            approved_repair |= repair_labels == component_id
    gap_mask = np.zeros_like(candidate_bool)
    observed_bottom = min(int(row_y.max()), candidate_bool.shape[0] - 1)
    for y in range(int(row_y.min()), observed_bottom + 1):
        left = int(round(np.interp(y, row_y, gap_left)))
        right = int(round(np.interp(y, row_y, gap_right)))
        gap_mask[y, max(x1, left):min(x2, right) + 1] = True
    result = (candidate_bool | approved_repair) & ~gap_mask
    return result, {"trouser_topology_applied": True}

def _fit_quadratic_border_background(lab, border, max_samples=25000):
    """Ước lượng nền studio trơn có gradient/vignette từ viền ảnh."""
    height, width = lab.shape[:2]
    yy, xx = np.indices((height, width), dtype=np.float32)
    x = 2.0 * xx / max(width - 1, 1) - 1.0
    y = 2.0 * yy / max(height - 1, 1) - 1.0
    border_x, border_y = x[border], y[border]
    border_lab = lab[border].astype(np.float32)
    if len(border_x) > max_samples:
        indices = np.linspace(0, len(border_x) - 1, max_samples, dtype=np.int64)
        border_x, border_y, border_lab = border_x[indices], border_y[indices], border_lab[indices]
    design = np.column_stack([np.ones_like(border_x), border_x, border_y, border_x * border_y, border_x**2, border_y**2]).astype(np.float32)
    coefficients, *_ = np.linalg.lstsq(design, border_lab, rcond=None)
    fit_residual = np.linalg.norm(design @ coefficients - border_lab, axis=1)
    inliers = fit_residual <= max(float(np.percentile(fit_residual, 85)), 2.0)
    if int(inliers.sum()) >= max(24, 4 * design.shape[1]):
        coefficients, *_ = np.linalg.lstsq(design[inliers], border_lab[inliers], rcond=None)
    features = (np.ones_like(x), x, y, x * y, x * x, y * y)
    background = np.empty_like(lab, dtype=np.float32)
    for channel in range(3):
        pred = sum(f * float(c) for f, c in zip(features, coefficients[:, channel]))
        background[:, :, channel] = np.clip(pred, 0.0, 255.0)
    return background

def _studio_background_cleanup(mask, image, label):
    """Bóc nền studio thông minh: khử bóng đổ (shadow-aware) và phục hồi vải."""
    rgb = np.asarray(image.convert("RGB") if hasattr(image, "convert") else image)[:, :, :3].astype(np.uint8)
    height, width = mask.shape
    if rgb.shape[:2] != (height, width):
        rgb = cv2.resize(rgb, (width, height), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    border_size = max(4, int(round(min(height, width) * 0.035)))
    border = np.zeros((height, width), dtype=bool)
    border[:border_size, :] = border[-border_size:, :] = border[:, :border_size] = border[:, -border_size:] = True
    border_pixels = lab[border]
    constant_background = np.median(border_pixels, axis=0)
    global_uniformity = float(np.percentile(np.linalg.norm(border_pixels - constant_background, axis=1), 90))
    quadratic_background = _fit_quadratic_border_background(lab, border)
    quadratic_distance_map = np.linalg.norm(lab - quadratic_background, axis=2)
    surface_residual_p90 = float(np.percentile(quadratic_distance_map[border], 90))
    use_quadratic = (global_uniformity > 14.0 and surface_residual_p90 <= 12.0 and surface_residual_p90 + 2.0 < global_uniformity)
    if global_uniformity > 20.0 and not use_quadratic:
        return mask.copy(), {"cleanup_applied": False, "skip_reason": "non_uniform_border"}
    background_map = quadratic_background if use_quadratic else np.broadcast_to(constant_background.reshape(1, 1, 3), lab.shape)
    distance_map = quadratic_distance_map if use_quadratic else np.linalg.norm(lab - background_map, axis=2)
    border_distances = quadratic_distance_map[border] if use_quadratic else np.linalg.norm(border_pixels - constant_background, axis=1)
    distance_threshold = float(np.clip(np.percentile(border_distances, 98) + 6.0, 10.0, 30.0))
    
    # Khử bóng đổ dựa trên sắc độ a/b (chroma)
    chroma_distance_map = np.linalg.norm(lab[:, :, 1:3] - background_map[:, :, 1:3], axis=2)
    chroma_threshold = float(np.clip(np.percentile(chroma_distance_map[border], 98) + 3.0, 4.0, 18.0))
    raw_chroma_distances = chroma_distance_map[mask.astype(bool)]
    garment_chroma_separation = float(np.median(raw_chroma_distances) if len(raw_chroma_distances) else 0.0)
    shadow_aware = garment_chroma_separation >= max(10.0, 1.35 * chroma_threshold)
    if shadow_aware:
        adaptive_chroma_threshold = max(chroma_threshold, min(18.0, 0.28 * garment_chroma_separation))
        image_foreground = (chroma_distance_map > adaptive_chroma_threshold) | (mask.astype(bool) & (distance_map > 0.55 * distance_threshold))
    else:
        image_foreground = distance_map > distance_threshold

    kernel_size = max(3, min(11, int(round(min(height, width) * 0.007)))) | 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    fg_u8 = cv2.morphologyEx(image_foreground.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    fg_u8 = cv2.morphologyEx(fg_u8, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    
    # Neo contour vào raw mask
    contours, _ = cv2.findContours(fg_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    selected_seed = np.zeros_like(fg_u8)
    raw_area = int(mask.sum())
    for contour in contours:
        if cv2.contourArea(contour) < max(32.0, raw_area * 0.0008):
            continue
        c_mask = np.zeros_like(fg_u8)
        cv2.drawContours(c_mask, [contour], -1, 1, thickness=cv2.FILLED)
        if int((c_mask.astype(bool) & mask).sum()) >= max(24, int(raw_area * 0.0015)):
            selected_seed |= fg_u8 & c_mask
    if not selected_seed.any():
        return mask.copy(), {"cleanup_applied": False, "skip_reason": "no_seed"}

    # Flood-fill vá lỗ nhỏ bên trong
    padded = np.pad(selected_seed, 1, mode="constant", constant_values=0).astype(np.uint8)
    flooded = padded.copy()
    cv2.floodFill(flooded, np.zeros((padded.shape[0] + 2, padded.shape[1] + 2), np.uint8), (0, 0), 1)
    enclosed = (flooded == 0).astype(np.uint8)
    hole_count, hole_labels, hole_stats, _ = cv2.connectedComponentsWithStats(enclosed, 8)
    small_holes = np.isin(hole_labels, [i for i in range(1, hole_count) if int(hole_stats[i, cv2.CC_STAT_AREA]) <= max(64, int(selected_seed.sum() * 0.005))])
    silhouette = (padded.astype(bool) | small_holes)[1:-1, 1:-1].astype(np.uint8)
    
    if label in {"trousers", "shorts"}:
        refined = silhouette.astype(bool)
    else:
        refined = cv2.dilate(silhouette, np.ones((3, 3), np.uint8), iterations=1) > 0
    refined, _ = _solidify_light_trousers(refined, mask, lab, label)
    
    # An toàn: Kiểm tra tỷ lệ giữ lại raw và mở rộng
    raw_retained = int((refined & mask).sum()) / max(raw_area, 1)
    area_ratio = int(refined.sum()) / max(raw_area, 1)
    added_fraction = int((refined & ~mask).sum()) / max(raw_area, 1)
    if label in UPPER_LABELS and added_fraction > 0.45:
        raw_support = cv2.dilate(mask.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21)), iterations=1).astype(bool)
        refined = mask.astype(bool) | (refined & raw_support)
    if raw_retained < 0.50 or area_ratio > 2.50:
        return mask.copy(), {"cleanup_applied": False, "skip_reason": "fail_closed_guard"}
    return refined, {"cleanup_applied": True, "shadow_aware": shadow_aware, "area_ratio": round(area_ratio, 3)}

def refine_garment_mask(raw_mask, label, image=None):
    raw = raw_mask.astype(bool)
    bg_cleaned, meta = _studio_background_cleanup(raw, image, label)
    comp_cleaned, _ = _remove_distant_dress_components(bg_cleaned, label)
    refined, _ = _dress_lower_branch_cleanup(comp_cleaned, label)
    return refined, meta

# ==============================================================================
# TIẾN HÀNH DỰ ĐOÁN & TINH CHỈNH MASK CHO VTON
# ==============================================================================
results = model.predict(source=garment_path, conf=0.25, verbose=False)
res = results[0]

orig_img = Image.open(garment_path).convert("RGB")
orig_np = np.array(orig_img)
h, w = orig_np.shape[:2]

if res.boxes is None or len(res.boxes) == 0:
    raise ValueError("DeepFashion2 không còn nhận diện được garment ở Bước 5. Hãy chạy lại Bước 4.")

box_confidences = res.boxes.conf.cpu().numpy()
box_classes = res.boxes.cls.cpu().numpy().astype(int)
detected_labels = [str(res.names[int(class_id)]) for class_id in box_classes]

# Ưu tiên 1: tìm detection có đúng nhãn garment_class_name (ví dụ: short_sleeved_shirt)
matching_indices = [
    index for index, label in enumerate(detected_labels)
    if label == garment_class_name
]

# Ưu tiên 2: nếu không có nhãn trùng khít, tìm detection cùng vùng category (ví dụ: cùng upper_body)
if not matching_indices and garment_category:
    matching_indices = [
        index for index, label in enumerate(detected_labels)
        if garment_category_from_label(label) == garment_category
    ]

candidate_indices = matching_indices or list(range(len(box_confidences)))
selected_index = max(candidate_indices, key=lambda index: float(box_confidences[index]))
selected_label = detected_labels[selected_index]
selected_confidence = float(box_confidences[selected_index])

# Chỉ ghi đè garment_class_name nếu KHÔNG có file tham chiếu product_analysis.md khóa trước
has_verified_reference = bool(globals().get("reference_metadata") and reference_metadata.get("resolved_label"))
if not has_verified_reference:
    garment_class_name = selected_label
    garment_description = selected_label.replace("_", " ")
    garment_category = garment_category_from_label(selected_label)
    if garment_category is None:
        raise ValueError(
            f"Mask được chọn có nhãn không hỗ trợ '{selected_label}'. "
            "Notebook đã dừng trước khi gọi GPU."
        )
    print(f"[*] Chọn mask '{selected_label}' có confidence={selected_confidence:.2f}")
else:
    print(f"🔒 [THAM CHIẾU] Giữ vững định tuyến: {garment_class_name} ({garment_category})")
    print(f"   (Dùng mask từ detection '{selected_label}' có conf={selected_confidence:.2f})")
print(f"[*] Định tuyến try-on theo vùng: {garment_category}")

if res.masks is not None and len(res.masks) > selected_index:
    mask_tensor = res.masks.data[selected_index].cpu().numpy()
    mask_resized = cv2.resize((mask_tensor * 255).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    raw_binary_mask = (mask_resized > 127).astype(np.uint8)
    if raw_binary_mask.sum() == 0:
        raise ValueError("Mask DeepFashion2 rỗng; không thể tách trang phục.")

    # ÁP DỤNG TINH CHỈNH STUDIO MASK & KHỬ BÓNG ĐỔ
    print("[*] Đang áp dụng thuật toán tinh chỉnh mask studio-aware & khử bóng đổ...")
    refined_mask_bool, refine_meta = refine_garment_mask(
        raw_binary_mask > 0,
        garment_class_name,
        image=orig_img
    )
    binary_mask = refined_mask_bool.astype(np.uint8)
    status_str = "Đã tinh chỉnh (khử bóng đổ & bảo toàn viền)" if refine_meta.get("cleanup_applied") else "Giữ mask gốc an toàn"
    print(f"[*] Kết quả bóc tách nền: {status_str} (area_ratio={refine_meta.get('area_ratio', 1.0)})")

    mask_y, mask_x = np.where(binary_mask > 0)
    if len(mask_x) == 0 or len(mask_y) == 0:
        binary_mask = raw_binary_mask
        mask_y, mask_x = np.where(binary_mask > 0)

    # Crop sát garment để loại người/nền khỏi đầu vào OOTDiffusion.
    padding = max(8, int(0.03 * max(w, h)))
    x1 = max(0, int(mask_x.min()) - padding)
    x2 = min(w, int(mask_x.max()) + padding + 1)
    y1 = max(0, int(mask_y.min()) - padding)
    y2 = min(h, int(mask_y.max()) + padding + 1)
    
    # Tạo ảnh RGBA trong suốt, làm mềm biên bằng Gaussian Blur
    rgba_full = np.zeros((h, w, 4), dtype=np.uint8)
    rgba_full[:, :, :3] = orig_np
    rgba_full[:, :, 3] = cv2.GaussianBlur(binary_mask * 255, (5, 5), 0)
    rgba = rgba_full[y1:y2, x1:x2]
    Image.fromarray(rgba).save(out_nobg_path)
    cv2.imwrite(out_mask_path, (binary_mask * 255).astype(np.uint8))

    alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
    isolated_tight = Image.fromarray(
        (rgba[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
    )

    # OOTDiffusion luôn resize garment về 768x1024. Đặt garment lên canvas 3:4 trước
    # để giữ nguyên tỷ lệ thật, tránh áo/váy bị kéo méo khi gửi tới Space.
    canvas_width, canvas_height = 768, 1024
    margin_ratio = 0.06
    usable_width = int(canvas_width * (1 - 2 * margin_ratio))
    usable_height = int(canvas_height * (1 - 2 * margin_ratio))
    fit_scale = min(
        usable_width / isolated_tight.width,
        usable_height / isolated_tight.height,
    )
    fitted_size = (
        max(1, int(round(isolated_tight.width * fit_scale))),
        max(1, int(round(isolated_tight.height * fit_scale))),
    )
    resampling = getattr(Image, "Resampling", Image).LANCZOS
    isolated_fitted = isolated_tight.resize(fitted_size, resampling)
    isolated_canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
    paste_xy = (
        (canvas_width - fitted_size[0]) // 2,
        (canvas_height - fitted_size[1]) // 2,
    )
    isolated_canvas.paste(isolated_fitted, paste_xy)
    isolated_canvas.save(out_isolated_path)
    isolated_rgb = np.asarray(isolated_canvas)
    garment_for_tryon_path = out_isolated_path
    print(f"✅ Đã tách riêng garment khỏi người/nền: {os.path.abspath(out_nobg_path)} ({os.path.getsize(out_nobg_path)/1024:.1f} KB)")
    print(f"✅ Đã tạo garment 768x1024, giữ đúng tỷ lệ trên nền trắng: {os.path.abspath(out_isolated_path)}")
    print(f"✅ Đã lưu mask phân đoạn nhị phân: {os.path.abspath(out_mask_path)} ({os.path.getsize(out_mask_path)/1024:.1f} KB)")
    
    # Trích xuất Color Palette (K-Means Clustering 3 màu)
    fg_pixels = orig_np[binary_mask > 0]
    if len(fg_pixels) > 50:
        sample_pixels = fg_pixels[::max(1, len(fg_pixels) // 1000)]
        kmeans = KMeans(n_clusters=min(3, len(sample_pixels)), random_state=42, n_init=5).fit(sample_pixels)
        colors = kmeans.cluster_centers_.astype(int)
        hex_codes = [f"#{c[0]:02X}{c[1]:02X}{c[2]:02X}" for c in colors]
        print(f"🎨 Bảng màu chủ đạo trích xuất: {hex_codes}")
    
    # Hiển thị trực quan 4 khung hình đối sánh
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    ax[0].imshow(orig_img); ax[0].set_title("1. Ảnh sản phẩm gốc")
    ax[1].imshow(raw_binary_mask, cmap="gray"); ax[1].set_title("2. Mask thô (DeepFashion2)")
    ax[2].imshow(binary_mask, cmap="gray"); ax[2].set_title("3. Mask đã tinh chỉnh khử bóng")
    ax[3].imshow(isolated_rgb); ax[3].set_title("4. Garment 768x1024 cho VTON")
    for a in ax: a.axis("off")
    plt.tight_layout()
    plt.show()
else:
    raise ValueError(
        f"DeepFashion2 nhận diện '{selected_label}' nhưng không tạo được segmentation mask. "
        "Không dùng threshold dự phòng vì có thể giữ lại người trong ảnh."
    )


### Bước 6: Thử đồ bằng OOTDiffusion VITON-HD/DressCode theo category

> Routing thử nghiệm: `upper_body → VITON-HD /process_hd`; `lower_body`, `dress → DressCode /process_dc`. Khung người được chia thành `full_body`, `three_quarter`, `half_body`; dress cho phép cả ba mức nhưng ảnh thiếu chân chạy ở chế độ giới hạn và có cảnh báo. Quần dài vẫn cần full-body; skirt/shorts chấp nhận ảnh 3/4 hoặc half-body ở chế độ giới hạn và có cảnh báo.

> Với `lower_body`, nhãn `skirt/shorts/trousers` do DeepFashion2 đọc từ ảnh output chỉ dùng để ưu tiên và cảnh báo, không chặn việc đổi subtype. Kết quả được ghép bằng mask của đồ cũ và đồ mới để giữ nguyên da chân, mặt, tóc, giày và nền ngoài vùng garment.

> Bộ chọn dùng hai tầng: trước hết giữ nhóm có mức khớp subtype tốt nhất; sau đó chấm điểm tổng hợp gồm màu garment, bảo toàn vùng ngoài garment, bảo toàn vùng cơ thể không được che, độ fitting hình học và confidence detector. Không metric đơn lẻ nào được tự quyết định kết quả.

> **Xác thực Hugging Face:** mở biểu tượng chìa khóa **Secrets** ở thanh trái Colab, tạo secret tên `HF_TOKEN`, dán token quyền `read`, rồi bật **Notebook access**. Không ghi token trực tiếp vào code.


In [ ]:
import time, os
from PIL import Image
import numpy as np
import cv2

from pathlib import Path


def prepare_oot_person(
    source_path,
    output_path,
    target_size=(768, 1024),
):
    image = Image.open(source_path).convert("RGB")

    original_width, original_height = image.size
    target_width, target_height = target_size

    # Scale theo cùng một tỷ lệ, không kéo méo ảnh.
    scale = min(
        target_width / original_width,
        target_height / original_height,
    )

    resized_width = round(original_width * scale)
    resized_height = round(original_height * scale)

    resized = image.resize(
        (resized_width, resized_height),
        Image.Resampling.LANCZOS,
    )

    # Lấy màu trung bình ở bốn góc làm màu padding.
    corners = [
        image.getpixel((0, 0)),
        image.getpixel((original_width - 1, 0)),
        image.getpixel((0, original_height - 1)),
        image.getpixel((original_width - 1, original_height - 1)),
    ]

    background_color = tuple(
        sum(pixel[channel] for pixel in corners) // len(corners)
        for channel in range(3)
    )

    canvas = Image.new(
        "RGB",
        target_size,
        background_color,
    )

    offset_x = (target_width - resized_width) // 2
    offset_y = (target_height - resized_height) // 2

    canvas.paste(resized, (offset_x, offset_y))
    canvas.save(output_path, "PNG")

    metadata = {
        "original_size": (original_width, original_height),
        "content_box": (
            offset_x,
            offset_y,
            offset_x + resized_width,
            offset_y + resized_height,
        ),
        "target_size": target_size,
    }

    return output_path, metadata


def restore_oot_result(output_image, metadata):
    output_image = output_image.convert("RGB")

    # Phòng trường hợp backend trả về kích thước khác.
    if output_image.size != metadata["target_size"]:
        output_image = output_image.resize(
            metadata["target_size"],
            Image.Resampling.LANCZOS,
        )

    # Bỏ vùng padding đã thêm trước inference.
    output_image = output_image.crop(metadata["content_box"])

    # Đưa kết quả trở lại đúng kích thước ảnh người ban đầu.
    return output_image.resize(
        metadata["original_size"],
        Image.Resampling.LANCZOS,
    )

print("=" * 70)
print("KHỞI CHẠY OOTDIFFUSION CHO MỌI CATEGORY")
print(f"[*] Ảnh người mẫu      : {person_path}")
print(f"[*] Ảnh garment đã tách : {garment_for_tryon_path}")
print("=" * 70)

start_time = time.time()
tryon_dir = "data/test_samples/results/tryon"
os.makedirs(tryon_dir, exist_ok=True)
tryon_out_path = os.path.join(tryon_dir, "tryon_result.png")

output = None
tryon_backend = None
remote_error = None

def load_hf_token():
    token = os.environ.get("HF_TOKEN")
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or token
    except Exception:
        # Secret chưa tồn tại/chưa cấp quyền hoặc đang chạy ngoài Colab.
        pass
    return token


hf_token = load_hf_token()
if hf_token:
    print("[*] Đã tìm thấy HF_TOKEN trong Colab Secrets; request sẽ được xác thực.")
else:
    print("[!] Chưa có HF_TOKEN; request ZeroGPU sẽ dùng quota ẩn danh rất thấp.")

def extract_gradio_image_paths(value):
    paths = []
    if isinstance(value, str):
        return [value]
    if isinstance(value, dict):
        for key in ("path", "image"):
            if key in value:
                paths.extend(extract_gradio_image_paths(value[key]))
    if isinstance(value, (tuple, list)):
        for item in value:
            paths.extend(extract_gradio_image_paths(item))
    return list(dict.fromkeys(paths))


def detect_output_garment(image, expected_category):
    """Nhận diện garment và trả cả mask vùng áo/quần; subtype chỉ là tín hiệu heuristic."""
    image = image.convert("RGB")
    image_np = np.asarray(image)
    result = fashion_detector.predict(
        source=image_np, conf=0.20, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0, None

    labels = [str(result.names[int(class_id)]) for class_id in result.boxes.cls.cpu().numpy()]
    confidences = result.boxes.conf.cpu().numpy()
    # matching_indices = [
    #     index for index, label in enumerate(labels)
    #     if garment_category_from_label(label) == expected_category
    # ]
    # if not matching_indices:
    #     return None, 0.0, None
    matching_indices = [
        index for index, label in enumerate(labels)
        if garment_category_from_label(label) == expected_category
    ]

    # Fallback cho Dress: Nếu là đầm màu sáng/kem tiệp màu nền, DeepFashion2 đôi khi
    # gán nhãn 'short_sleeved_shirt' hoặc 'upper_body' nhưng mask trải dài xuống đùi.
    if not matching_indices and expected_category == "dress" and result.masks is not None:
        for idx, (cls_id, conf) in enumerate(zip(result.boxes.cls.cpu().numpy(), confidences)):
            lbl = str(result.names[int(cls_id)])
            # Nếu nhận diện thành áo hoặc chân váy nhưng che phủ diện tích lớn ở thân
            if any(k in lbl for k in ["shirt", "top", "dress", "skirt"]) and conf >= 0.15:
                matching_indices.append(idx)

    if not matching_indices:
        return None, 0.0, None


    best_index = max(matching_indices, key=lambda index: float(confidences[index]))
    # Chỉ lấy mask của detection tốt nhất. Union mọi detection cùng category có thể
    # trộn mask 'skirt' với detection phụ 'trousers' và vô tình cho phép AI sửa cả chân.
    region_mask = np.zeros(image_np.shape[:2], dtype=np.uint8)
    if result.masks is not None and len(result.masks.data) > best_index:
        raw_mask = result.masks.data[best_index].cpu().numpy()
        resized_mask = cv2.resize(
            raw_mask, (image.width, image.height), interpolation=cv2.INTER_NEAREST
        )
        region_mask = (resized_mask > 0.5).astype(np.uint8) * 255
    else:
        x1, y1, x2, y2 = result.boxes.xyxy[best_index].cpu().numpy().astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(image.width, x2), min(image.height, y2)
        region_mask[y1:y2, x1:x2] = 255

    return labels[best_index], float(confidences[best_index]), region_mask


def refine_garment_mask_with_color(image, coarse_mask, pose_context=None):
    """
    Tinh chỉnh mask trang phục người mẫu bằng phân tích phân bố màu (Lab Color Space):
    1. Nhận diện chính xác màu trang phục người mẫu đang mặc so với màu nền phòng/studio.
    2. Loại bỏ các phần nền studio bị YOLO-seg/DeepFashion2 nhận diện lấn ra ngoài.
    3. Bảo tồn đường cong eo tự nhiên mà không cần hiển thị mã màu ra giao diện.
    """
    if coarse_mask is None or np.count_nonzero(coarse_mask > 127) < 100:
        return coarse_mask

    img_np = np.asarray(image.convert("RGB"))
    h, w = img_np.shape[:2]
    binary_mask = (coarse_mask > 127).astype(np.uint8)

    # Lấy vùng lõi chắc chắn là trang phục (core_garment)
    core_mask = cv2.erode(binary_mask, np.ones((11, 11), np.uint8), iterations=2)
    if np.count_nonzero(core_mask) < 200:
        core_mask = cv2.erode(binary_mask, np.ones((5, 5), np.uint8), iterations=1)
    if np.count_nonzero(core_mask) < 100:
        return coarse_mask

    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)
    garment_color = np.median(lab[core_mask > 0], axis=0)

    bg_samples = []
    if pose_context and "box" in pose_context:
        bx1, by1, bx2, by2 = [int(v) for v in pose_context["box"]]
        if bx1 > 30:
            bg_left = lab[max(0, by1):min(h, by2), max(0, bx1 - 80):max(0, bx1 - 10)]
            if bg_left.size > 0:
                bg_samples.append(bg_left.reshape(-1, 3))
        if bx2 < w - 30:
            bg_right = lab[max(0, by1):min(h, by2), min(w, bx2 + 10):min(w, bx2 + 80)]
            if bg_right.size > 0:
                bg_samples.append(bg_right.reshape(-1, 3))

    if not bg_samples:
        corner_pixels = np.concatenate([
            lab[:40, :40].reshape(-1, 3),
            lab[:40, w-40:].reshape(-1, 3),
            lab[h-40:, :40].reshape(-1, 3),
            lab[h-40:, w-40:].reshape(-1, 3),
        ], axis=0)
        bg_samples.append(corner_pixels)

    all_bg_pixels = np.concatenate(bg_samples, axis=0)
    bg_color = np.median(all_bg_pixels, axis=0)

    dist_to_garment = np.linalg.norm(lab - garment_color, axis=2)
    dist_to_bg = np.linalg.norm(lab - bg_color, axis=2)

    is_confident_bg = (dist_to_bg < 32.0) & (dist_to_garment > dist_to_bg * 1.35)

    refined_mask = binary_mask.copy()
    refined_mask[is_confident_bg] = 0
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    return (refined_mask * 255).astype(np.uint8)


def detect_waist_background_gaps(image, pose_context):
    """
    Nhận diện khoảng trống nền tự nhiên giữa eo và cánh tay buông xuôi của người mẫu.
    Vùng khoảng trống này sẽ được bảo vệ tuyệt đối để đầm mới không phình đè lên đường thắt eo.
    """
    if pose_context is None or "keypoints" not in pose_context:
        return None
    kpts = pose_context["keypoints"]
    confs = pose_context.get("confidences")
    if len(kpts) < 13:
        return None
    if confs is not None and min(confs[[5, 6, 11, 12]]) < 0.20:
        return None

    img_np = np.asarray(image.convert("RGB"))
    h, w = img_np.shape[:2]
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)

    bx1, by1, bx2, by2 = [int(v) for v in pose_context["box"]]
    bg_samples = []
    if bx1 > 20:
        bg_samples.append(lab[max(0, by1):min(h, by2), max(0, bx1 - 60):max(0, bx1 - 5)].reshape(-1, 3))
    if bx2 < w - 20:
        bg_samples.append(lab[max(0, by1):min(h, by2), min(w, bx2 + 5):min(w, bx2 + 60)].reshape(-1, 3))
    if not bg_samples:
        return None
    bg_color = np.median(np.concatenate(bg_samples, axis=0), axis=0)

    shoulder_y = float(np.mean(kpts[[5, 6], 1]))
    hip_y = float(np.mean(kpts[[11, 12], 1]))
    torso_span = max(20.0, hip_y - shoulder_y)

    y_start = int(round(shoulder_y + 0.30 * torso_span))
    y_end = int(round(hip_y + 0.10 * torso_span))

    waist_gap_mask = np.zeros((h, w), dtype=bool)
    dist_to_bg = np.linalg.norm(lab[y_start:y_end] - bg_color, axis=2)
    is_bg_waist = dist_to_bg < 30.0

    if len(kpts) >= 8 and (confs is None or confs[7] >= 0.25):
        el_x = int(round(kpts[7][0]))
        hip_lx = int(round(kpts[11][0]))
        x_min, x_max = min(el_x, hip_lx), max(el_x, hip_lx)
        if x_max - x_min > 5:
            waist_gap_mask[y_start:y_end, x_min:x_max] = is_bg_waist[:, x_min:x_max]

    if len(kpts) >= 9 and (confs is None or confs[8] >= 0.25):
        er_x = int(round(kpts[8][0]))
        hip_rx = int(round(kpts[12][0]))
        x_min, x_max = min(er_x, hip_rx), max(er_x, hip_rx)
        if x_max - x_min > 5:
            waist_gap_mask[y_start:y_end, x_min:x_max] = is_bg_waist[:, x_min:x_max]

    return waist_gap_mask


def garment_edit_mask(source_mask, candidate_mask, waist_gap_mask=None):
    """Hợp mask đồ cũ/mới và làm mềm biên để giới hạn vùng được phép thay đổi."""
    available_masks = [mask for mask in (source_mask, candidate_mask) if mask is not None]
    if not available_masks:
        return None

    edit_mask = np.maximum.reduce(available_masks).astype(np.uint8)
    edit_mask = cv2.dilate(edit_mask, np.ones((11, 11), np.uint8), iterations=1)

    # Bảo toàn khoảng trống thắt eo: không cho đầm mới lan vào khoảng hở giữa eo và cánh tay
    if waist_gap_mask is not None and np.any(waist_gap_mask):
        edit_mask[waist_gap_mask] = 0

    return cv2.GaussianBlur(edit_mask, (0, 0), sigmaX=3, sigmaY=3)


def outside_garment_change_score(candidate_image, source_image, edit_mask):
    """Điểm càng thấp thì mặt, da, chân và nền ngoài vùng thử đồ càng được giữ nguyên."""
    if edit_mask is None:
        return float("inf")
    candidate_np = np.asarray(candidate_image.convert("RGB"), dtype=np.float32)
    source_np = np.asarray(source_image.convert("RGB"), dtype=np.float32)
    outside = edit_mask < 8
    if not np.any(outside):
        return float("inf")
    pixel_difference = np.mean(np.abs(candidate_np - source_np), axis=2)
    return float(np.mean(pixel_difference[outside]))


def change_to_preservation_score(change_score, scale=25.0):
    if not np.isfinite(change_score):
        return 0.0
    return float(np.exp(-max(0.0, change_score) / scale))


def composite_preserving_source(
    candidate_image, source_image, edit_mask, protected_mask=None
):
    candidate_np = np.asarray(candidate_image.convert("RGB"), dtype=np.float32)
    source_np = np.asarray(source_image.convert("RGB"), dtype=np.float32)
    if edit_mask is None:
        alpha = np.ones((*source_np.shape[:2], 1), dtype=np.float32)
    else:
        alpha = edit_mask.astype(np.float32)[:, :, None] / 255.0
    if protected_mask is not None:
        protection = protected_mask.astype(np.float32)[:, :, None] / 255.0
        alpha *= 1.0 - protection
    protected = candidate_np * alpha + source_np * (1.0 - alpha)
    return Image.fromarray(np.clip(protected, 0, 255).astype(np.uint8))


def detect_person_pose_context(image):
    result = person_detector.predict(
        source=np.asarray(image.convert("RGB")), conf=0.20, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
        return None
    boxes = result.boxes.xyxy.cpu().numpy()
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    best_index = int(np.argmax(areas))
    keypoints = result.keypoints.xy[best_index].cpu().numpy()
    if result.keypoints.conf is None:
        confidences = np.where(np.any(keypoints > 0, axis=1), 1.0, 0.0)
    else:
        confidences = result.keypoints.conf[best_index].cpu().numpy()

    # Nhận diện tư thế tay: phát hiện tay giấu sau lưng hoặc bị khuất
    hands_behind_back = False
    if len(keypoints) >= 11 and len(confidences) >= 11:
        wrist_confs = confidences[[9, 10]]
        if np.mean(wrist_confs) < 0.35 or (confidences[9] < 0.35 and confidences[10] < 0.35):
            hands_behind_back = True
        elif len(keypoints) >= 13 and min(confidences[[11, 12]]) >= 0.25:
            hip_x1 = min(keypoints[11][0], keypoints[12][0])
            hip_x2 = max(keypoints[11][0], keypoints[12][0])
            hip_span = max(10.0, hip_x2 - hip_x1)
            wrists_behind = (hip_x1 - 0.25 * hip_span <= keypoints[9][0] <= hip_x2 + 0.25 * hip_span) and                             (hip_x1 - 0.25 * hip_span <= keypoints[10][0] <= hip_x2 + 0.25 * hip_span)
            if wrists_behind and np.mean(wrist_confs) < 0.55:
                hands_behind_back = True

    return {
        "box": boxes[best_index].astype(np.float32),
        "keypoints": keypoints.astype(np.float32),
        "confidences": confidences.astype(np.float32),
        "hands_behind_back": hands_behind_back,
    }


def upper_body_protected_mask(image_size, pose_context, target_label):
    """Bảo vệ phần cánh tay mà vest/áo ngắn tay không được phép tự vẽ thêm vải."""
    if pose_context is None:
        return None
    if pose_context.get("hands_behind_back", False):
        return None
    label = (target_label or "").lower()
    if "long_sleeved" in label:
        return None

    width, height = image_size
    keypoints = pose_context["keypoints"]
    confidences = pose_context["confidences"]
    if len(keypoints) < 11:
        return None

    shoulder_span = abs(float(keypoints[6][0] - keypoints[5][0]))
    thickness = max(12, min(42, int(round(max(60.0, shoulder_span) * 0.18))))
    start_fraction = 0.15 if any(name in label for name in ("vest", "sling")) else 0.45
    protected = np.zeros((height, width), dtype=np.uint8)

    for shoulder_index, elbow_index, wrist_index in ((5, 7, 9), (6, 8, 10)):
        if min(confidences[[shoulder_index, elbow_index, wrist_index]]) < 0.25:
            continue
        shoulder = keypoints[shoulder_index]
        elbow = keypoints[elbow_index]
        wrist = keypoints[wrist_index]
        start = shoulder + start_fraction * (elbow - shoulder)
        start_xy = tuple(np.rint(start).astype(int))
        elbow_xy = tuple(np.rint(elbow).astype(int))
        wrist_xy = tuple(np.rint(wrist).astype(int))
        cv2.line(protected, start_xy, elbow_xy, 255, thickness, cv2.LINE_AA)
        cv2.line(protected, elbow_xy, wrist_xy, 255, thickness, cv2.LINE_AA)
        cv2.circle(protected, elbow_xy, thickness // 2, 255, -1, cv2.LINE_AA)
        cv2.circle(protected, wrist_xy, thickness // 2, 255, -1, cv2.LINE_AA)

    if not np.any(protected):
        return None
    return cv2.GaussianBlur(protected, (0, 0), sigmaX=4, sigmaY=4)


def protected_region_artifact_score(candidate_image, source_image, protected_mask):
    """Trả (tỷ lệ artefact nặng, điểm bảo toàn) trên vùng da tay cần giữ nguyên."""
    if protected_mask is None:
        return 0.0, 1.0
    candidate_np = np.asarray(candidate_image.convert("RGB"), dtype=np.float32)
    source_np = np.asarray(source_image.convert("RGB"), dtype=np.float32)
    weights = protected_mask.astype(np.float32) / 255.0
    if float(np.sum(weights)) < 1.0:
        return 0.0, 1.0

    channel_difference = np.mean(np.abs(candidate_np - source_np), axis=2)
    source_luma = 0.299 * source_np[:, :, 0] + 0.587 * source_np[:, :, 1] + 0.114 * source_np[:, :, 2]
    candidate_luma = 0.299 * candidate_np[:, :, 0] + 0.587 * candidate_np[:, :, 1] + 0.114 * candidate_np[:, :, 2]

    # Chỉ coi là artefact khi pixel tối sụp mạnh hoặc đổi màu cực lớn.
    # Sai khác ánh sáng/texture nhỏ do diffusion không được phép loại mọi candidate.
    severe_artifact = ((source_luma - candidate_luma) > 50.0) | (channel_difference > 70.0)
    artifact_ratio = float(np.sum(weights * severe_artifact) / np.sum(weights))
    preservation = float(np.exp(-artifact_ratio / 0.08))
    return artifact_ratio, preservation


def garment_fit_score(candidate_mask, pose_context, expected_category, target_label):
    """Heuristic 0..1 phát hiện garment lệch tâm, tràn thân hoặc phình bất thường."""
    if candidate_mask is None or pose_context is None:
        return 0.5
    binary = candidate_mask > 127
    ys, xs = np.where(binary)
    if len(xs) < 50:
        return 0.0

    x1, x2 = float(xs.min()), float(xs.max() + 1)
    y1, y2 = float(ys.min()), float(ys.max() + 1)
    person_x1, person_y1, person_x2, person_y2 = pose_context["box"]
    person_width = max(1.0, float(person_x2 - person_x1))
    center_offset = abs(((x1 + x2) / 2.0) - ((person_x1 + person_x2) / 2.0)) / person_width
    outside_ratio = float(
        np.mean((xs < person_x1 - 5) | (xs > person_x2 + 5) | (ys < person_y1 - 5) | (ys > person_y2 + 5))
    )
    penalty = 0.35 * min(1.0, center_offset / 0.22) + 0.25 * min(1.0, outside_ratio / 0.05)

    if expected_category == "upper_body":
        keypoints = pose_context["keypoints"]
        confidences = pose_context["confidences"]
        if min(confidences[[5, 6, 11, 12]]) >= 0.25:
            shoulder_span = max(1.0, abs(float(keypoints[6][0] - keypoints[5][0])))
            shoulder_y = float(np.mean(keypoints[[5, 6], 1]))
            hip_y = float(np.mean(keypoints[[11, 12], 1]))
            torso_height = max(1.0, hip_y - shoulder_y)
            width_ratio = (x2 - x1) / shoulder_span
            height_ratio = (y2 - y1) / torso_height
            long_sleeved = "long_sleeved" in (target_label or "").lower()
            max_width_ratio = 3.0 if long_sleeved else 1.9
            max_height_ratio = 2.2 if long_sleeved else 1.45
            penalty += 0.20 * min(1.0, max(0.0, width_ratio - max_width_ratio) / 0.8)
            penalty += 0.20 * min(1.0, max(0.0, height_ratio - max_height_ratio) / 0.8)

    return float(np.clip(1.0 - penalty, 0.0, 1.0))

def calculate_limb_integrity_score(candidate_mask, pose_context, garment_label):
    """
    Chấm điểm độ toàn vẹn chi thể (0.0 - 1.0):
    Phát hiện và phạt nặng candidate nếu áo try-on phình ra che mất cổ tay hoặc cẳng tay
    đang đút vào túi quần của người mẫu.
    """
    if candidate_mask is None or pose_context is None:
        return 1.0

    keypoints = pose_context.get("keypoints")
    confidences = pose_context.get("confidences")
    if keypoints is None or confidences is None or len(keypoints) < 11:
        return 1.0

    mask = (candidate_mask > 127).astype(np.uint8)
    h, w = mask.shape[:2]

    shoulder_span = abs(float(keypoints[6][0] - keypoints[5][0]))
    wrist_radius = max(14, int(round(max(60.0, shoulder_span) * 0.08)))
    forearm_thickness = max(12, int(round(max(60.0, shoulder_span) * 0.09)))

    label = (garment_label or "").lower()
    # Đối với đầm liền (dress/skirt):
    # Tùng váy/chân váy (xoè, chữ A, xếp ly) phủ từ eo/hông xuống đùi/gối.
    # Tay người mẫu khi buông thõng tự nhiên sẽ nằm ngang hoặc trước tùng váy, khiến
    # mask 2D của đầm luôn trùng toạ độ (overlap) với cổ tay. Đây là đặc trưng hình học tự nhiên
    # của đầm, không phải áo phình che cổ tay. Độ phình ngang của đầm đã có evaluate_dress_proportions_and_color kiểm soát.
    if any(word in label for word in ["dress", "skirt", "dam", "vay"]):
        return 1.0

    allows_forearm_coverage = any(
        word in label for word in ["long_sleeved", "coat", "jacket", "sweater"]
    )
    # Với trang phục dài tay, mask phủ khuỷu/cổ tay là đúng thiết kế chứ không phải artefact.
    if allows_forearm_coverage:
        return 1.0
    is_short_sleeve = True

    total_penalty = 0.0

    # Kiểm tra cả 2 tay: (elbow, wrist) = (7, 9) tay trái, (8, 10) tay phải
    for elbow_idx, wrist_idx in ((7, 9), (8, 10)):
        # 1. Kiểm tra vùng cổ tay (người mẫu đút tay vào túi quần)
        if confidences[wrist_idx] >= 0.25:
            wx, wy = int(round(keypoints[wrist_idx][0])), int(round(keypoints[wrist_idx][1]))
            wrist_roi = np.zeros((h, w), dtype=np.uint8)
            cv2.circle(wrist_roi, (wx, wy), wrist_radius, 255, -1)

            overlap_pixels = np.count_nonzero((wrist_roi > 0) & (mask > 0))
            roi_pixels = np.count_nonzero(wrist_roi > 0)
            if roi_pixels > 0:
                overlap_ratio = overlap_pixels / roi_pixels
                if overlap_ratio > 0.05:
                    # Phạt tỷ lệ thuận với diện tích áo trùm lên cổ tay
                    total_penalty += min(1.0, overlap_ratio * 2.0) * 0.50

        # 2. Với áo ngắn tay/polo: kiểm tra nửa dưới cẳng tay (từ 35% khuỷu tay đến cổ tay)
        if is_short_sleeve and confidences[elbow_idx] >= 0.25 and confidences[wrist_idx] >= 0.25:
            ex, ey = keypoints[elbow_idx]
            wx, wy = keypoints[wrist_idx]
            p_start = (int(round(ex + 0.35 * (wx - ex))), int(round(ey + 0.35 * (wy - ey))))
            p_end = (int(round(wx)), int(round(wy)))

            forearm_roi = np.zeros((h, w), dtype=np.uint8)
            cv2.line(forearm_roi, p_start, p_end, 255, forearm_thickness, cv2.LINE_AA)

            forearm_overlap = np.count_nonzero((forearm_roi > 0) & (mask > 0))
            forearm_total = np.count_nonzero(forearm_roi > 0)
            if forearm_total > 0:
                forearm_ratio = forearm_overlap / forearm_total
                if forearm_ratio > 0.05:
                    # Phạt nếu vạt áo/tay áo phình sang hai bên đè lên cẳng tay
                    total_penalty += min(1.0, forearm_ratio * 2.0) * 0.50

    return float(np.clip(1.0 - total_penalty, 0.0, 1.0))
def evaluate_dress_proportions_and_color(
    candidate_image, candidate_mask, pose_context, reference_image, reference_mask, source_garment_mask=None
):
    """
    Tiêu chí chuyên biệt cho Dress:
    1. Đo độ vừa vặn (Fit): Phạt nặng nếu váy phình to bất thường so với khoảng cách hai hông
       hoặc phình to hơn đáng kể so với trang phục gốc người mẫu đang mặc.
    2. Phạt nếu váy bị loang mảng màu lạ không có trong đầm mẫu (chống hallucination).
    """
    if candidate_mask is None or pose_context is None:
        return 0.5, 0.5

    keypoints = pose_context.get("keypoints")
    confidences = pose_context.get("confidences")
    if keypoints is None or len(keypoints) < 13:
        return 0.5, 0.5

    mask = (candidate_mask > 127).astype(np.uint8)
    ys, xs = np.where(mask > 0)
    if len(xs) < 100:
        return 0.0, 0.0

    # 1. Đo tỷ lệ phình ngang của váy so với khoảng cách hai hông
    # Keypoint 11, 12 là Left Hip và Right Hip
    hip_width = abs(float(keypoints[12][0] - keypoints[11][0]))
    if hip_width < 20.0:
        # Nếu hông bị che hoặc đứng nghiêng 3/4, dùng 75% bề rộng vai
        hip_width = max(30.0, abs(float(keypoints[6][0] - keypoints[5][0])) * 0.75)

    dress_width = float(xs.max() - xs.min())
    width_ratio = dress_width / hip_width

    # Tỷ lệ lý tưởng cho đầm suông/chữ A thường trong khoảng 1.1 - 1.35 lần hông
    fit_penalty = 0.0
    if width_ratio > 1.35:
        fit_penalty += min(1.0, (width_ratio - 1.35) / 0.5) * 0.60

    # 2. So sánh trực tiếp với bề rộng trang phục gốc của người mẫu (nếu có)
    # Tránh đầm sau VTON phình to quá khổ so với cơ thể thật
    if source_garment_mask is not None:
        s_mask = (source_garment_mask > 127).astype(np.uint8)
        s_ys, s_xs = np.where(s_mask > 0)
        if len(s_xs) >= 100:
            source_width = float(s_xs.max() - s_xs.min())
            if source_width > 20.0:
                excess_ratio = (dress_width - source_width) / source_width
                # Nếu đầm mới phình to hơn 18% so với đồ gốc người mẫu đang mặc
                if excess_ratio > 0.18:
                    fit_penalty += min(1.0, (excess_ratio - 0.18) / 0.35) * 0.50

    dress_fit_score = float(np.clip(1.0 - fit_penalty, 0.0, 1.0))

    # 3. Kiểm tra độ tương đồng màu sắc & chống loang màu lạ
    color_score = garment_color_similarity(
        reference_image, reference_mask, candidate_image, candidate_mask
    )

    return dress_fit_score, color_score

def garment_color_similarity(reference_image, reference_mask, candidate_image, candidate_mask):
    """So màu trong vùng garment bằng histogram Lab; 1.0 là gần màu gốc nhất."""
    if reference_mask is None or candidate_mask is None:
        return 0.0

    def masked_lab_histogram(image, mask):
        image_np = np.asarray(image.convert("RGB"), dtype=np.uint8)
        mask = (mask > 127).astype(np.uint8) * 255
        # Co mask nhẹ để da/nền ở rìa garment không làm lệch màu.
        clean_mask = cv2.erode(mask, np.ones((5, 5), np.uint8), iterations=1)
        if np.count_nonzero(clean_mask) < 50:
            clean_mask = mask
        if np.count_nonzero(clean_mask) < 50:
            return None
        lab_image = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
        histogram = cv2.calcHist(
            [lab_image], [0, 1, 2], clean_mask,
            [8, 12, 12], [0, 256, 0, 256, 0, 256],
        )
        return cv2.normalize(
            histogram, None, alpha=1.0, norm_type=cv2.NORM_L1
        )

    reference_histogram = masked_lab_histogram(reference_image, reference_mask)
    candidate_histogram = masked_lab_histogram(candidate_image, candidate_mask)
    if reference_histogram is None or candidate_histogram is None:
        return 0.0

    distance = cv2.compareHist(
        reference_histogram, candidate_histogram, cv2.HISTCMP_BHATTACHARYYA
    )
    return float(np.clip(1.0 - distance, 0.0, 1.0))


DRESS_STYLE_FAMILY = {
    "vest_dress": "sleeveless_dress",
    "sling_dress": "sleeveless_dress",
    "short_sleeved_dress": "short_sleeved_dress",
    "long_sleeved_dress": "long_sleeved_dress",
}


def candidate_match_quality(candidate_label, target_label, expected_category):
    """Trả mức khớp subtype; confidence chỉ được dùng khi hai candidate cùng mức."""
    candidate_label = (candidate_label or "").lower()
    target_label = (target_label or "").lower()
    if not candidate_label:
        return 0, "không nhận diện đúng category"
    if candidate_label == target_label:
        return 3, "khớp subtype chính xác"

    if expected_category == "dress":
        candidate_family = DRESS_STYLE_FAMILY.get(candidate_label)
        target_family = DRESS_STYLE_FAMILY.get(target_label)
        if candidate_family and candidate_family == target_family:
            return 2, f"cùng nhóm kiểu tay ({target_family})"
        return 1, "đúng dress nhưng sai kiểu tay/subtype"

    if expected_category == "lower_body":
        # DeepFashion2 có thể nhận lại quần/shorts đang mặc hoặc nhầm giữa skirt,
        # shorts và trousers trên ảnh người. Đây vẫn là đúng vùng lower-body nên
        # chỉ hạ ưu tiên và cảnh báo, không loại candidate khỏi kết quả try-on.
        return (
            1,
            f"đúng lower-body nhưng detector nhận '{candidate_label}' thay vì '{target_label}'",
        )

    return 1, "đúng upper-body nhưng khác subtype"


# Cấu hình ổn định dùng chung cho hai checkpoint OOTDiffusion VITON-HD/DressCode.
OOT_N_SAMPLES = 4
OOT_N_STEPS = 20
OOT_IMAGE_SCALE = 1.6  # Cân bằng guidance để đầm ôm vừa vặn theo đường cong cơ thể, chống phình to/thùng thình
OOT_SEED = 42  # Cố định để so sánh thí nghiệm; đổi thành -1 nếu muốn biến thể ngẫu nhiên.
OOT_MANUAL_CANDIDATE = None  # Có thể đặt 1..4 sau khi xem bảng candidate rồi chạy lại cell.

print(
    f"[*] Cấu hình OOTDiffusion: samples={OOT_N_SAMPLES}, steps={OOT_N_STEPS}, "
    f"scale={OOT_IMAGE_SCALE}, seed={OOT_SEED}"
)

def validate_person_frame_for_category(selected_frame, selected_category, garment_subtype):
    """Áp dụng progressive validation: ưu tiên cảnh báo thay vì chặn khi vẫn có thể thử đồ."""
    if selected_frame == "invalid":
        raise ValueError("Ảnh không thấy rõ vai và hông nên chưa đủ điều kiện try-on.")

    if selected_category == "upper_body":
        return "accepted", "Khung ảnh đủ cho thử áo."

    if selected_category == "dress":
        if selected_frame == "full_body":
            return "accepted", "Ảnh toàn thân phù hợp để thử mọi chiều dài đầm."
        if selected_frame == "three_quarter":
            return (
                "accepted_with_warning",
                "Ảnh 3/4 vẫn được thử đầm; phù hợp nhất với đầm ngắn hoặc đầm trên gối. "
                "Phần đầm dài dưới vùng ảnh có thể không được thể hiện đầy đủ.",
            )
        return (
            "accepted_with_warning",
            "Ảnh chỉ thấy đến hông/đùi nhưng vẫn được thử đầm ở chế độ giới hạn. "
            "Kết quả chỉ đáng tin trong vùng cơ thể đang nhìn thấy; nên dùng ảnh thấy ít nhất đến đầu gối "
            "nếu muốn đánh giá phom và chiều dài đầm.",
        )

    # Lower-body: shorts/skirt có thể chạy giới hạn với ảnh 3/4 hoặc half-body; trousers vẫn cần thấy mắt cá chân.
    subtype = (garment_subtype or "").lower()
    if selected_frame == "full_body":
        return "accepted", "Ảnh toàn thân phù hợp để thử lower-body."
    if selected_frame in {"three_quarter", "half_body"} and subtype in {"shorts", "skirt"}:
        if selected_frame == "three_quarter":
            message = (
                f"Ảnh 3/4 đủ để thử {subtype}, nhưng chỉ nên đánh giá phần trang phục nằm trong vùng ảnh."
            )
        else:
            message = (
                f"Ảnh half-body vẫn được thử {subtype} ở chế độ giới hạn vì đã thấy hông/đùi. "
                "Kết quả chỉ đáng tin trong vùng cơ thể đang hiển thị; nên dùng ảnh thấy đầu gối để đánh giá "
                "đầy đủ chiều dài và phom trang phục."
            )
        return "accepted_with_warning", message
    raise ValueError(
        f"Ảnh '{selected_frame}' chưa đủ vùng chân để thử '{subtype or selected_category}'. "
        "Quần dài cần thấy mắt cá; chân váy/quần short cần thấy ít nhất một đầu gối."
    )


frame_validation_status, frame_validation_message = validate_person_frame_for_category(
    person_frame, garment_category, garment_class_name
)
if frame_validation_status == "accepted_with_warning":
    print(f"[!] Khung người được chấp nhận có điều kiện: {frame_validation_message}")
else:
    print(f"[+] Kiểm tra khung người/category hợp lệ: {frame_validation_message}")

if garment_category == "dress":
    print(
        "[i] Hướng dẫn thử đầm: có thể dùng ảnh đang mặc áo + quần, nhưng nên chọn "
        "trang phục ôm gọn (áo hai dây/áo ôm, legging, jean ôm, short ôm hoặc đồ tập).\n"
        "    ⚠️ LƯU Ý TƯ THẾ (Pose): Khuyến nghị người mẫu đứng thẳng, HAI TAY BUÔNG TỰ NHIÊN HAI BÊN.\n"
        "    Tránh giấu tay ra sau lưng hoặc khoanh tay trước ngực, vì mô hình khuếch tán OOTDiffusion\n"
        "    khi inpaint vùng trang phục trống có xu hướng tái tạo cánh tay ra phía trước theo catalog chuẩn."
    )
elif garment_category == "lower_body":
    lower_guidance = {
        "shorts": (
            "nên mặc short ôm hoặc legging ngắn; tránh quần dài, váy dài và tay che hông"
        ),
        "skirt": (
            "nên mặc short/legging ôm hoặc chân váy gọn có chiều dài tương tự; "
            "tránh quần ống rộng, váy xòe nhiều lớp và tay che eo"
        ),
        "trousers": (
            "nên mặc legging, jean ôm hoặc quần dáng gọn; tránh váy dài, quần ống rộng "
            "và chân bắt chéo"
        ),
    }.get(garment_class_name, "nên mặc phần dưới ôm gọn và không che hông/chân")
    print(f"[i] Hướng dẫn thử lower-body: {lower_guidance}.")

# Routing OOTDiffusion: VITON-HD xử lý upper-body; DressCode xử lý lower-body và dress.
# Frame chỉ quyết định mức chấp nhận/cảnh báo, không tự đổi endpoint.
try:
    from gradio_client import Client, handle_file

    def create_space_client(space_id):
        if not hf_token:
            return Client(space_id)
        try:
            return Client(space_id, token=hf_token)
        except TypeError:
            # Tương thích gradio_client phiên bản cũ (trước khi hf_token đổi tên thành token).
            return Client(space_id, hf_token=hf_token)

    client = create_space_client("levihsu/OOTDiffusion")
    prepared_person_path = str(
        Path(tryon_dir) / "person_oot_768x1024.png"
    )

    prepared_person_path, person_resize_metadata = prepare_oot_person(
        source_path=person_path,
        output_path=prepared_person_path,
    )
    prepared_person_image = Image.open(prepared_person_path).convert("RGB")

    # Ảnh/mask tham chiếu dùng để đo độ trung thành màu của từng candidate.
    if os.path.exists(out_nobg_path):
        garment_reference_rgba = Image.open(out_nobg_path).convert("RGBA")
        garment_reference_image = garment_reference_rgba.convert("RGB")
        garment_reference_mask = np.asarray(
            garment_reference_rgba.getchannel("A"), dtype=np.uint8
        )
    else:
        garment_reference_image = Image.open(garment_for_tryon_path).convert("RGB")
        _, _, garment_reference_mask = detect_output_garment(
            garment_reference_image, garment_category
        )

    pose_context = detect_person_pose_context(prepared_person_image)
    if pose_context and pose_context.get("hands_behind_back"):
        print(
            "⚠️ [CẢNH BÁO TƯ THẾ]: Phát hiện hai tay người mẫu để ra sau lưng.\n"
            "   Mô hình khuếch tán OOTDiffusion (huấn luyện trên catalog chuẩn) có xu hướng tự vẽ thêm\n"
            "   cánh tay ra phía trước khi inpaint vùng thân người trống.\n"
            "   Hệ thống đã tự động vô hiệu hoá composite đè vải cũ để tránh vệt trắng mờ trên đầm mới."
        )
    protected_body_mask = None
    if garment_category in {"upper_body", "dress"}:
        protected_body_mask = upper_body_protected_mask(
            prepared_person_image.size, pose_context, garment_class_name
        )

    source_garment_mask = None
    if garment_category == "dress":
        source_masks = []
        for source_category in ("dress", "upper_body", "lower_body"):
            _, _, detected_mask = detect_output_garment(
                prepared_person_image, source_category
            )
            if detected_mask is not None:
                source_masks.append(detected_mask)
        if source_masks:
            source_garment_mask = np.maximum.reduce(source_masks).astype(np.uint8)
    else:
        _, _, source_garment_mask = detect_output_garment(
            prepared_person_image, garment_category
        )

    # Tinh chỉnh mask đồ đang mặc bằng phân tích phân bố màu sắc & khóa khoảng hở eo-tay
    waist_gap_mask = None
    if source_garment_mask is not None:
        source_garment_mask = refine_garment_mask_with_color(
            prepared_person_image, source_garment_mask, pose_context
        )
    if pose_context is not None:
        waist_gap_mask = detect_waist_background_gaps(prepared_person_image, pose_context)

    if garment_category in {"upper_body", "lower_body", "dress"}:
        if source_garment_mask is None:
            print("[!] Không tách được mask đồ đang mặc; điểm bảo toàn ngoài garment sẽ giảm độ tin cậy.")
        else:
            refine_note = " (đã tinh chỉnh theo màu đồ gốc để ôm sát eo)"
            if waist_gap_mask is not None and np.any(waist_gap_mask):
                refine_note += " & đã khóa khoảng hở eo-tay"
            print(f"[+] Đã tạo mask đồ đang mặc{refine_note} để bảo toàn cơ thể và nền ngoài garment.")

    def run_ootdiffusion(selected_category):
        common_inputs = {
            "vton_img": handle_file(prepared_person_path),
            "garm_img": handle_file(garment_for_tryon_path),
            "n_samples": OOT_N_SAMPLES,
            "n_steps": OOT_N_STEPS,
            "image_scale": OOT_IMAGE_SCALE,
            "seed": OOT_SEED,
        }

        if selected_category == "upper_body":
            backend = "OOTDiffusion VITON-HD (Upper-body)"
            print(f"[*] {selected_category} -> {backend} [/process_hd]")
            # /process_hd cố định upper-body và không nhận tham số category.
            response = client.predict(
                **common_inputs,
                api_name="/process_hd",
            )
            return response, backend

        oot_category = {
            "lower_body": "Lower-body",
            "dress": "Dress",
        }[selected_category]
        backend = f"OOTDiffusion DressCode ({oot_category})"
        print(f"[*] {selected_category} -> {backend} [/process_dc]")
        response = client.predict(
            **common_inputs,
            category=oot_category,
            api_name="/process_dc",
        )
        return response, backend

    result, tryon_backend = run_ootdiffusion(garment_category)

    result_img_paths = extract_gradio_image_paths(result)
    if not result_img_paths:
        raise RuntimeError(f"Không đọc được đường dẫn ảnh từ kết quả Gradio: {result!r}")

    candidate_reports = []
    candidate_images = []
    eligible_candidates = []
    for candidate_index, candidate_path in enumerate(result_img_paths, start=1):
        try:
            candidate_image = Image.open(candidate_path).convert("RGB")
        except Exception as img_err:
            print(f"⚠️ Bỏ qua candidate #{candidate_index} do file ảnh không hợp lệ/lỗi ({img_err}): {candidate_path}")
            continue
        candidate_label, candidate_confidence, candidate_garment_mask = detect_output_garment(
            candidate_image, garment_category
        )
        match_level, match_reason = candidate_match_quality(
            candidate_label, garment_class_name, garment_category
        )
        color_similarity = garment_color_similarity(
            garment_reference_image, garment_reference_mask,
            candidate_image, candidate_garment_mask,
        )
        edit_mask = None
        outside_change = 0.0
        outside_preservation = 1.0
        body_change = 0.0
        body_preservation = 1.0
        
        protected_candidate_image = candidate_image
        if garment_category in {"upper_body", "lower_body", "dress"}:
            # Chỉ lấy vùng garment mới từ candidate; mọi pixel ngoài vùng đồ lấy từ ảnh người gốc.
            edit_mask = garment_edit_mask(
                source_garment_mask, candidate_garment_mask, waist_gap_mask=waist_gap_mask
            )
            outside_change = outside_garment_change_score(
                candidate_image, prepared_person_image, edit_mask
            )
            outside_preservation = change_to_preservation_score(outside_change)
            # Pixel-wise body comparison chỉ phù hợp với dress. VITON-HD upper-body có thể
            # dịch chuyển tay vài pixel dù kết quả hợp lệ; dùng metric này làm hard gate sẽ loại nhầm.
            if garment_category == "dress":
                body_change, body_preservation = protected_region_artifact_score(
                    candidate_image, prepared_person_image, protected_body_mask
                )
            # VITON-HD đã sinh upper-body hoàn chỉnh. Composite với union mask áo cũ/mới
            # sẽ kéo tay áo và vạt áo cũ trở lại, tạo các mảng trắng chồng lên áo mới.
            # Vì vậy upper-body giữ nguyên candidate thô; lower-body/dress mới cần composite.
            # Với upper_body và dress: OOTDiffusion đã sinh trang phục hoàn chỉnh và đồng nhất.
            # KHÔNG truyền protected_body_mask vào composite để tránh kéo mảng màu/vải áo cũ
            # từ prepared_person_image đè lên váy/áo mới (gây vệt trắng mờ lem luốc trên vải).
            # composite_preserving_source chỉ dùng edit_mask để bảo toàn tuyệt đối nền, đầu, chân ngoài vùng đồ.
            if garment_category != "upper_body":
                protected_candidate_image = composite_preserving_source(
                    candidate_image, prepared_person_image, edit_mask,
                    protected_mask=None,
                )

        fit_score = garment_fit_score(candidate_garment_mask, pose_context, garment_category, garment_class_name)
        # Tính điểm bảo toàn chi thể (chống che mất tay đút túi quần)
        limb_integrity = calculate_limb_integrity_score(
            candidate_garment_mask, pose_context, garment_class_name
        )

        # if garment_category == "upper_body":
        #     # Ưu tiên cao nhất (35%) cho độ toàn vẹn chi thể không bị áo che khuất
        #     quality_score = (
        #         0.25 * color_similarity
        #         + 0.35 * limb_integrity        # <--- TIÊU CHÍ MỚI
        #         + 0.25 * fit_score
        #         + 0.15 * candidate_confidence
        #     )
        # else:
        #     quality_score = (
        #         0.20 * color_similarity
        #         + 0.25 * outside_preservation
        #         + 0.25 * body_preservation
        #         + 0.25 * fit_score
        #         + 0.05 * candidate_confidence
        #     )
        dress_width_fit = 1.0
        if garment_category == "dress":
            dress_width_fit, color_similarity = evaluate_dress_proportions_and_color(
                candidate_image, candidate_garment_mask, pose_context,
                garment_reference_image, garment_reference_mask,
                source_garment_mask=source_garment_mask
            )
            # Dress phải giữ nguyên tay/cơ thể và nền; màu đẹp không được bù cho artefact ngoài garment.
            quality_score = (
                0.30 * color_similarity
                + 0.15 * dress_width_fit
                + 0.15 * outside_preservation
                + 0.15 * body_preservation
                + 0.10 * limb_integrity
                + 0.10 * fit_score
                + 0.05 * candidate_confidence
            )
        elif garment_category == "upper_body":
            # Đánh giá độ dài tay áo so với người mẫu shop (nếu có tham chiếu)
            sleeve_fidelity = 1.0
            if globals().get("shop_model_fit") and shop_model_fit.get("sleeve_ratio"):
                target_sleeve = shop_model_fit["sleeve_ratio"]
                cand_fit = analyze_model_garment_fit(candidate_image, "upper_body")
                if cand_fit and cand_fit.get("sleeve_ratio"):
                    # Thưởng candidate có độ dài tay áo vươn dài sát với mẫu shop, phạt candidate bị co cụt tay áo
                    sleeve_diff = abs(cand_fit["sleeve_ratio"] - target_sleeve)
                    sleeve_fidelity = float(np.clip(1.0 - sleeve_diff * 1.5, 0.25, 1.0))
            
            quality_score = (
                0.20 * color_similarity
                + 0.25 * limb_integrity
                + 0.25 * fit_score
                + 0.20 * sleeve_fidelity
                + 0.10 * candidate_confidence
            )
        else: # lower_body
            quality_score = (
                0.20 * color_similarity
                + 0.25 * outside_preservation
                + 0.25 * body_preservation
                + 0.25 * fit_score
                + 0.05 * candidate_confidence
            )

        artifact_reasons = []
        outside_minimum = {"upper_body": 0.72, "lower_body": 0.65, "dress": 0.65}[garment_category]
        if outside_preservation < outside_minimum:
            artifact_reasons.append(
                f"ngoài garment thấp ({outside_preservation:.3f} < {outside_minimum:.2f})"
            )
        # Với dress, composite_preserving_source đã tự động khôi phục da tay/cơ thể từ ảnh gốc.
        # Chỉ loại nếu vùng cánh tay bị hỏng/đen cực nặng (< 0.005)
        if garment_category == "dress" and body_preservation < 0.005:
            artifact_reasons.append(f"tay/cơ thể biến dạng nặng ({body_preservation:.3f})")
        # limb_minimum chỉ áp dụng cho upper_body (chống vạt áo phình đè mất tay đút túi quần)
        limb_minimum = {"upper_body": 0.65}.get(garment_category)
        if limb_minimum is not None and limb_integrity < limb_minimum:
            artifact_reasons.append(
                f"mask garment lấn vào tay ({limb_integrity:.3f} < {limb_minimum:.2f})"
            )
        artifact_ok = not artifact_reasons
        artifact_status = "đạt artifact gate" if artifact_ok else "LOẠI: " + "; ".join(artifact_reasons)

        candidate_reports.append(
            f"#{candidate_index}: detector_label={candidate_label or 'không nhận diện'} "
            f"({candidate_confidence:.2f}; {match_reason}; "
            f"màu={color_similarity:.3f}; ngoài={outside_preservation:.3f}; "
            f"cơ thể={body_preservation:.3f}; fit={fit_score:.3f}; "
            f"tổng={quality_score:.3f}; {artifact_status})"
        )
        print(f"    Candidate {candidate_reports[-1]}")

        candidate_info = {
            "index": candidate_index,
            "image": protected_candidate_image,
            "raw_image": candidate_image,
            "label": candidate_label,
            "confidence": candidate_confidence,
            "color_similarity": color_similarity,
            "limb_integrity": limb_integrity,    # <--- THÊM DÒNG NÀY
            "outside_change": outside_change,
            "outside_preservation": outside_preservation,
            "body_change": body_change,
            "body_preservation": body_preservation,
            "fit_score": fit_score,
            "quality_score": quality_score,
            "match_level": match_level,
            "match_reason": match_reason,
            "artifact_ok": artifact_ok,
            "artifact_status": artifact_status,
            "eligible": match_level > 0 and artifact_ok,
        }
        candidate_images.append(candidate_info)
        if candidate_info["eligible"]:
            eligible_candidates.append(candidate_info)

    if not eligible_candidates:
        print(
            f"⚠️ CẢNH BÁO: Không candidate nào vượt qua 100% hard-gate nghiêm ngặt cho '{garment_category}'.\n"
            f"   Hệ thống kích hoạt Fallback: Tự động chọn candidate có quality_score cao nhất "
            f"để hoàn thành try-on và hiển thị ảnh kết quả."
        )
        valid_type_candidates = [c for c in candidate_images if c.get("match_level", 0) > 0]
        selection_pool = valid_type_candidates if valid_type_candidates else candidate_images
        selected_candidate = max(
            selection_pool,
            key=lambda item: (item["quality_score"], item["confidence"])
        )
        selected_candidate["artifact_status"] += " (FALLBACK: chọn do tổng điểm cao nhất)"
    elif OOT_MANUAL_CANDIDATE is not None:
        manual_matches = [
            item for item in eligible_candidates
            if item["index"] == OOT_MANUAL_CANDIDATE
        ]
        if not manual_matches:
            manual_matches = [
                item for item in candidate_images
                if item["index"] == OOT_MANUAL_CANDIDATE
            ]
        if not manual_matches:
            raise ValueError(
                f"OOT_MANUAL_CANDIDATE={OOT_MANUAL_CANDIDATE} không hợp lệ hoặc sai category."
            )
        selected_candidate = manual_matches[0]
    else:
        # Với Dress: ưu tiên ứng viên có màu sắc trung thực với đầm shop và form dáng vừa vặn
        if garment_category == "dress":
            selected_candidate = max(
                eligible_candidates,
                key=lambda item: item["quality_score"]
            )
        else:
            best_match_level = max(item["match_level"] for item in eligible_candidates)
            selection_pool = [
                item for item in eligible_candidates
                if item["match_level"] == best_match_level
            ]
            selected_candidate = max(
                selection_pool,
                key=lambda item: (item["quality_score"], item["confidence"])
            )


    output = selected_candidate["image"]
    selected_index = selected_candidate["index"]
    print(
        f"[+] Chọn candidate #{selected_index}: detector_label={selected_candidate['label']} "
        f"(tổng={selected_candidate['quality_score']:.3f}, "
        f"màu={selected_candidate['color_similarity']:.3f}, "
        f"ngoài={selected_candidate['outside_preservation']:.3f}, "
        f"cơ thể={selected_candidate['body_preservation']:.3f}, "
        f"fit={selected_candidate['fit_score']:.3f}, "
        f"confidence={selected_candidate['confidence']:.2f})."
    )
    print(f"    Lý do chọn: {selected_candidate['match_reason']}.")
    if selected_candidate["match_level"] < 3:
        print(
            f"[!] Không có candidate khớp chính xác '{garment_class_name}'; "
            "đang dùng phương án gần nhất. Hãy kiểm tra trực quan trước khi chấp nhận."
        )

    # Luôn hiển thị toàn bộ candidate để có thể đánh giá artefact bằng mắt.
    fig, axes = plt.subplots(1, len(candidate_images), figsize=(5 * len(candidate_images), 6))
    axes = np.atleast_1d(axes).ravel()
    for axis, item in zip(axes, candidate_images):
        chosen = item["index"] == selected_index
        exact_match = item["match_level"] == 3
        chosen_text = " - ĐÃ CHỌN" if chosen and exact_match else (" - TẠM CHỌN" if chosen else "")
        # Hiển thị candidate thô để đánh giá đúng ảnh do backend sinh trước hậu xử lý.
        axis.imshow(item["image"])
        axis.set_title(
            f"Candidate #{item['index']}{chosen_text}" +
            f"\nDetector: {item['label'] or 'không nhận diện'} ({item['confidence']:.2f})" +
            f"\nTổng: {item['quality_score']:.3f} | Màu: {item['color_similarity']:.3f}" +
            f"\nNgoài: {item['outside_preservation']:.3f} | Cơ thể: {item['body_preservation']:.3f} | Fit: {item['fit_score']:.3f}" +
            f"\nTay: {item['limb_integrity']:.3f} | Conf: {item['confidence']:.2f}" +
            f"\n{item['artifact_status']}",
            color=("green" if exact_match else "darkorange") if chosen else "black",
            fontweight="bold" if chosen else "normal",
        )

        axis.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"[+] {tryon_backend} sinh ảnh thành công!")
except Exception as e:
    remote_error = e
    print(f"[-] Backend try-on cho '{garment_category}' thất bại: {e}")

if output is None:
    raise RuntimeError(
        f"Không thể chạy OOTDiffusion cho {garment_category}: {remote_error}. "
        "Notebook đã dừng để không lưu ảnh preview giả làm kết quả VTON. "
        "Nếu request chưa sinh được ảnh, hãy kiểm tra HF_TOKEN/quota; nếu các candidate đều bị loại, "
        "hãy xem lý do quality/artifact gate ở log rồi chạy lại Bước 6."
    )

infer_time = time.time() - start_time

# Khôi phục đúng tỷ lệ/kích thước ảnh người trước khi lưu và lập báo cáo.
output = restore_oot_result(
    output,
    person_resize_metadata,
)
# LƯU RIÊNG ẢNH KẾT QUẢ THỬ ĐỒ (ẢNH 1/2 TRONG results/tryon/)
output.save(tryon_out_path, "PNG")
print(f"[+] Hoàn thành thử đồ AI sau: {infer_time:.2f} giây!")
print(f"✅ [1/2] Đã lưu RIÊNG ảnh thử đồ ảo (Single Try-on) vào:")
print(f"    -> {os.path.abspath(tryon_out_path)} ({os.path.getsize(tryon_out_path)/1024:.1f} KB)")

# Chuẩn bị các biến cho Bước 7
human_img = Image.open(person_path).convert("RGB")
cloth_img = Image.open(garment_for_tryon_path).convert("RGB")


### Bước 7: Xuất bảng ảnh đối sánh nghiệm thu Tuần 4 & Ảnh thử đồ AI sau VTON riêng lẻ


In [ ]:
import matplotlib.pyplot as plt

report_compare_path = "data/test_samples/results/tryon/week4_report_comparison.jpg"
os.makedirs(os.path.dirname(report_compare_path), exist_ok=True)

# 1. Bảng đối sánh kết quả nghiệm thu (4 cột nếu có người mẫu shop, 3 cột nếu không có)
has_shop_model = bool(globals().get("shop_model_path") and os.path.exists(shop_model_path))
if has_shop_model:
    shop_model_img = Image.open(shop_model_path).convert("RGB")
    fig, ax = plt.subplots(1, 4, figsize=(22, 7))
    ax[0].imshow(shop_model_img); ax[0].set_title("1. Người mẫu shop (Form chuẩn)", fontsize=13, fontweight="bold")
    ax[1].imshow(cloth_img); ax[1].set_title("2. Trang phục shop (YoFaSo)", fontsize=13, fontweight="bold")
    ax[2].imshow(human_img); ax[2].set_title("3. Khách hàng gốc (User)", fontsize=13, fontweight="bold")
    ax[3].imshow(output); ax[3].set_title(f"4. Kết quả thử đồ AI ({tryon_backend} - {infer_time:.2f}s)", fontsize=13, fontweight="bold")
    for a in ax: a.axis('off')
    plt.tight_layout()
    plt.savefig(report_compare_path, dpi=200, bbox_inches='tight')
    plt.show()
else:
    fig, ax = plt.subplots(1, 3, figsize=(18, 8))
    ax[0].imshow(human_img); ax[0].set_title("1. Người mẫu gốc (YoFaSo User)", fontsize=14, fontweight="bold")
    ax[1].imshow(cloth_img); ax[1].set_title("2. Trang phục shop (YoFaSo Garment)", fontsize=14, fontweight="bold")
    ax[2].imshow(output); ax[2].set_title(f"3. Kết quả thử đồ AI ({tryon_backend} - {infer_time:.2f}s)", fontsize=14, fontweight="bold")
    for a in ax: a.axis('off')
    plt.tight_layout()
    plt.savefig(report_compare_path, dpi=200, bbox_inches='tight')
    plt.show()

# 2. Hiển thị THÊM ảnh kết quả thử đồ AI sau VTON riêng lẻ (Single Try-on Result)
fig_single, ax_single = plt.subplots(figsize=(8, 10))
ax_single.imshow(output)
ax_single.set_title(f"Ảnh kết quả sau VTON riêng lẻ ({tryon_backend} - {infer_time:.2f}s)", fontsize=14, fontweight="bold", pad=12)
ax_single.axis('off')
plt.tight_layout()
plt.show()

print("="*75)
print("             TỔNG KẾT KẾT QUẢ THỬ NGHIỆM TUẦN 4 (YoFaSoShop_VTON)             ")
print("="*75)
print(f"- Trạng thái bóc tách nền: THÀNH CÔNG (DeepFashion2 YOLOv8-seg + Studio-Aware Refinement)")
print(f"- Mô hình Pose ước lượng khung người & tứ chi: YOLO26s-Pose (Ultralytics SOTA)")
print(f"- Backend try-on: {tryon_backend}")
print(f"- Category: {garment_category}; khung người: {person_frame}; xác thực: {frame_validation_status}; thời gian render: {infer_time:.2f} giây")
print(f"- Thư mục results/segmented/: Đã lưu garment mask, RGBA và garment nền trắng")
print(f"- Thư mục results/tryon/    : Đã lưu đủ 2 ảnh kết quả:")
print(f"    [1] Ảnh thử đồ đơn lẻ : {os.path.abspath(tryon_out_path)}")
print(f"    [2] Bảng đối sánh 3 cột: {os.path.abspath(report_compare_path)}")
print("="*75)


### Bước 8: Tải trọn bộ kết quả về máy tính cá nhân (Dành riêng cho Google Colab)
Nếu bạn chạy trên **Google Colab**, toàn bộ ảnh được lưu trữ trên máy chủ đám mây của Google. Chạy cell bên dưới để tự động nén toàn bộ thư mục `results/` và tải file `.zip` về thư mục **Downloads** trên máy tính của bạn!

In [ ]:
import shutil, os

results_dir = "data/test_samples/results"
zip_base_name = "yofaso_vton_week4_results"
zip_file = f"{zip_base_name}.zip"

# Nén toàn bộ thư mục kết quả
if os.path.exists(results_dir):
    shutil.make_archive(zip_base_name, 'zip', results_dir)
    print(f"📦 Đã nén thành công: {zip_file} ({os.path.getsize(zip_file)/1024:.1f} KB)")
    
    # Kích hoạt tải về tự động nếu chạy trên Colab
    try:
        from google.colab import files
        print("🚀 Đang gửi yêu cầu tải file zip về trình duyệt của bạn...")
        files.download(zip_file)
        print("👉 Bạn chỉ cần kiểm tra thư mục Downloads trên máy tính và giải nén dán vào data/test_samples/results/")
    except ImportError:
        print(f"💻 Bạn đang chạy cục bộ (Local). Toàn bộ ảnh đã được lưu trực tiếp tại:")
        print(f"   {os.path.abspath(results_dir)}")
else:
    print(f"⚠️ Chưa tìm thấy thư mục {results_dir} để đóng gói!")
